# CS-781 Phase 2 L-NER: Uploaded Hybrid Decoder + LoRMA + Knowledge Distillation

**Base model:** the Phase-1 model implementation is copied verbatim from your uploaded `v4(1).ipynb` architecture cell. I verified the source code cell matches exactly before adding a Phase-2-only `forward_hidden` helper that adds no parameters and does not alter checkpoint keys.

**Important architecture-label clarification:** the filename `v4(1).ipynb` suggests V4, but the notebook's own architecture section and saved checkpoint config explicitly label this as **V2 / `lightweight_hybrid_decoder`**. The actual architecture is a 12-layer decoder-only Transformer with hybrid global/local causal attention, RMSNorm, RoPE, GQA, GELU FFN, and KV cache. This Phase-2 experiment uses the exact supplied architecture and checkpoint; it does not relabel it as another architecture.

## Planned experiments
1. **LoRMA adaptation:** freeze the Phase-1 hybrid backbone, add multiplicative low-rank adapters on the query/value projections, and train a character-boundary BIO head.
2. **Teacher-student knowledge distillation:** use the LoRMA-adapted model as teacher and a smaller six-layer student initialized from selected layers of the same Phase-1 checkpoint. Train the student with gold-label loss plus temperature-scaled teacher-logit distillation.

These experiments use the capstone's mandatory LoRMA and knowledge-distillation techniques. This notebook covers this supplied hybrid architecture only; the course-wide requirement to compare experiments across all four Phase-1 architectures must be handled by the corresponding notebooks for your other models.

The official metric is strict span Macro-F1: start offset, end offset, and label must all match exactly. The pipeline includes character-offset integrity checks, document-disjoint splits, overlap-window target masking, complete character coverage audits, overlap-deduplicated class weights, one-character span support, correct partial gradient-accumulation groups, validation-only calibration, and one final ZIP.

**No score is guaranteed.** Static/synthetic tests do not replace loading your actual checkpoint and running validation/training on Kaggle.

**Required inputs:** the exact checkpoint saved by your upload (normally `hybrid_model.pt`), the matching `bpe_en_25k.json`, and the Phase-2 train/test JSON files. If both `hybrid_model.pt` and `v4_model.pt` are available, set `HYBRID_CHECKPOINT_PATH` explicitly to your intended checkpoint.


## 1. Runtime, reproducibility, paths, and compute settings

The checkpoint is loaded strictly **before** any LoRMA wrappers are inserted. Do not point this notebook at `v1_model.pt` or `v2_model.pt`.

In [ ]:
import os, json, re, math, time, random, copy, zipfile, hashlib, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF") or HF_TOKEN
except Exception:
    pass

# Set explicit paths here if automatic discovery finds multiple or no files.
HYBRID_CHECKPOINT_PATH = ""  # e.g. /kaggle/input/your-checkpoint-dataset/hybrid_model.pt
BPE_PATH = ""            # e.g. /kaggle/input/bpe-tokenizer/bpe_en_25k.json
PHASE2_ROOT_DIR = "/kaggle/working/cs781_phase2_uploaded_hybrid_lorma_kd"
CHECKPOINT_DIR = os.path.join(PHASE2_ROOT_DIR, "checkpoints")
OUTPUT_DIR = os.path.join(PHASE2_ROOT_DIR, "artifacts")
os.makedirs(CHECKPOINT_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_TOKEN_LENGTH = 512
WINDOW_OVERLAP = 256
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.08
MAX_GRAD_NORM = 1.0
FOCAL_GAMMA = 1.0

# Teacher: PEFT adaptation, stop early on held-out strict span Macro-F1.
LORMA_RANK = 8
LORMA_ALPHA = 16.0
LORMA_DROPOUT = 0.05
LORMA_LR = 1.5e-4
CHAR_HEAD_LR = 3e-4
TEACHER_EPOCHS = 5
TEACHER_PATIENCE = 2

# Student: six-layer hybrid student, hard gold labels + teacher soft logits.
STUDENT_BACKBONE_LR = 3e-5
STUDENT_EPOCHS = 4
STUDENT_PATIENCE = 2
KD_TEMPERATURE = 2.0
KD_HARD_WEIGHT = 0.6
KD_SOFT_WEIGHT = 0.4
assert abs(KD_HARD_WEIGHT + KD_SOFT_WEIGHT - 1.0) < 1e-8

print("Output root:", PHASE2_ROOT_DIR)
print("Uploaded-hybrid LoRMA epochs <=", TEACHER_EPOCHS, "| student KD epochs <=", STUDENT_EPOCHS)

## 2. Load the exact BPE-25k tokenizer

The offset encoder below mirrors the custom whitespace + byte-BPE implementation. It refuses to continue if its token IDs diverge from the tokenizer used to train uploaded Phase-1 hybrid checkpoint.

In [ ]:
BPE_FILENAME = "bpe_en_25k.json"
if not BPE_PATH:
    candidates = []
    for search_root in ("/kaggle/input", "/kaggle/working"):
        if os.path.isdir(search_root):
            for root, _, files in os.walk(search_root):
                if BPE_FILENAME in files:
                    candidates.append(os.path.join(root, BPE_FILENAME))
    candidates = sorted(set(candidates))
    if len(candidates) == 1:
        BPE_PATH = candidates[0]
    elif len(candidates) > 1:
        preferred = [p for p in candidates if "bpe_en_25k.json" in p]
        if len(preferred) == 1:
            BPE_PATH = preferred[0]
        else:
            raise RuntimeError("Multiple tokenizer files found. Set BPE_PATH explicitly:\n" + "\n".join(candidates))
if not BPE_PATH or not os.path.isfile(BPE_PATH):
    raise FileNotFoundError("Could not find bpe_en_25k.json. Attach the matching tokenizer as a Kaggle input or set BPE_PATH.")

with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)
bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
assert int(bpe_data["vocab_size"]) == 25000 and len(bpe_itos) == 25000

bpe_stoi = {token: idx for idx, token in enumerate(bpe_itos)}
merge_to_id = {}
for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge
    merge_to_id[(bpe_stoi[left_token], bpe_stoi[right_token])] = 261 + merge_index

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos, self.stoi = itos, stoi
        self.merge_to_id, self.merges = merge_to_id, merges
        self.vocab_size = len(itos)
        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        sequence = [4 + byte for byte in word.encode("utf-8")]
        sequence.append(self.word_boundary_id)
        while len(sequence) > 1:
            best_position, best_token_id = None, None
            for i in range(len(sequence) - 1):
                new_token_id = self.merge_to_id.get((sequence[i], sequence[i + 1]))
                if new_token_id is not None and (best_token_id is None or new_token_id < best_token_id):
                    best_position, best_token_id = i, new_token_id
            if best_position is None:
                break
            i = best_position
            sequence = sequence[:i] + [best_token_id] + sequence[i + 2:]
        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        ids = [self.bos_token_id] if add_bos else []
        for word in text.split():
            ids.extend(self._encode_word(word))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def _token_to_bytes(self, token_id):
        token = self.itos[int(token_id)]
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])
        if token == "<wb>":
            return b" "
        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""
        if token.startswith("<bpe_"):
            merge_index = int(token_id) - 261
            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]
                return self._token_to_bytes(self.stoi[left_token]) + self._token_to_bytes(self.stoi[right_token])
        return b""

    def decode(self, token_ids):
        output = bytearray()
        for token_id in token_ids:
            if 0 <= int(token_id) < self.vocab_size:
                output.extend(self._token_to_bytes(int(token_id)))
        return bytes(output).decode("utf-8", errors="replace")

tokenizer = ByteLevelBPE25k(bpe_itos, bpe_stoi, merge_to_id, bpe_merges)
assert tokenizer.vocab_size == 25000
print("Tokenizer loaded:", BPE_PATH)
print("Tokenizer round trip:", tokenizer.decode(tokenizer.encode("Supreme Court of India")))

## 3. Resolve the official Phase-2 files

Use the attached Kaggle dataset when available; otherwise download from the course dataset repository.

In [ ]:
from huggingface_hub import hf_hub_download

PHASE2_HF_PATHS = {
    "fold1-text.json": "Phase_2_LNER/Train/fold1-text.json",
    "fold1-gold.json": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2-text.json": "Phase_2_LNER/Train/fold2-text.json",
    "fold2-gold.json": "Phase_2_LNER/Train/fold2-gold.json",
    "fold3-text.json": "Phase_2_LNER/Test/fold3-text.json",
    "labels.txt": "Phase_2_LNER/Train/labels.txt",
}
PHASE2_FILES = {}
for filename, hub_filename in PHASE2_HF_PATHS.items():
    local_matches = []
    for search_root in ("/kaggle/input", "/kaggle/working"):
        if os.path.isdir(search_root):
            for root, _, files in os.walk(search_root):
                if filename in files:
                    local_matches.append(os.path.join(root, filename))
    local_matches = sorted(set(local_matches))
    if len(local_matches) == 1:
        PHASE2_FILES[filename] = local_matches[0]
    elif len(local_matches) > 1:
        # Prefer a path explicitly under Phase_2_LNER if there is one unique match.
        preferred = [p for p in local_matches if "Phase_2_LNER" in p]
        if len(preferred) == 1:
            PHASE2_FILES[filename] = preferred[0]
        else:
            raise RuntimeError(f"Multiple local copies of {filename}; resolve explicitly: {local_matches}")
    else:
        PHASE2_FILES[filename] = hf_hub_download(
            repo_id=HF_DATASET_REPO, filename=hub_filename,
            repo_type="dataset", token=HF_TOKEN or None
        )
    print(filename, "->", PHASE2_FILES[filename])

## 4. Load annotations, validate the data, and construct a document-level label-aware split

Text/gold IDs, offsets, labels, and training/test separation are checked before training. Validation labels are used only for model selection/calibration, never as training targets.

In [ ]:
from collections import Counter

EXPECTED_PHASE2_LABELS = {
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
}


def _load_json_file(filename):
    with open(PHASE2_FILES[filename], "r", encoding="utf-8") as handle:
        return json.load(handle)


def _normalise_text_map(raw_data, filename):
    # The Phase 2 specification describes text files as document-ID -> full text mappings.
    if not isinstance(raw_data, dict):
        raise TypeError(f"{filename} should contain a JSON object keyed by document ID; got {type(raw_data).__name__}.")
    result = {}
    for document_id, value in raw_data.items():
        if isinstance(value, str):
            text_value = value
        elif isinstance(value, dict) and isinstance(value.get("text"), str):
            # Supports a wrapped record if the published file uses that variant.
            text_value = value["text"]
        else:
            raise TypeError(
                f"Unexpected text record for document {document_id!r} in {filename}: {type(value).__name__}."
            )
        result[str(document_id)] = text_value
    return result


def _first_present(record, keys, field_name, document_id):
    for key in keys:
        if key in record:
            return record[key]
    raise KeyError(
        f"Missing {field_name!r} field for document {document_id!r}. Expected one of {keys}; got keys {list(record)}."
    )


def _normalise_entities_for_document(raw_entities, document_id, filename):
    # Supports the published [start, end, label, optional_text] format and dictionary variants.
    if raw_entities is None:
        return []
    if isinstance(raw_entities, dict):
        if "entities" in raw_entities:
            raw_entities = raw_entities["entities"]
        elif any(k in raw_entities for k in ("start", "start_char", "start_offset")):
            raw_entities = [raw_entities]
        else:
            raise TypeError(
                f"Unexpected annotation object for document {document_id!r} in {filename}: {list(raw_entities)[:12]}"
            )
    if not isinstance(raw_entities, (list, tuple)):
        raise TypeError(
            f"Annotations for document {document_id!r} in {filename} should be a list; got {type(raw_entities).__name__}."
        )

    entities = []
    for entity_index, entity in enumerate(raw_entities):
        if isinstance(entity, dict):
            start = _first_present(entity, ("start", "start_char", "start_offset", "begin"), "start", document_id)
            end = _first_present(entity, ("end", "end_char", "end_offset", "stop"), "end", document_id)
            label = _first_present(entity, ("label", "entity_label", "type", "entity_type"), "label", document_id)
            entity_text = entity.get("text", entity.get("entity_text", entity.get("surface")))
        elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
            start, end, label = entity[:3]
            entity_text = entity[3] if len(entity) >= 4 else None
        else:
            raise TypeError(
                f"Unexpected entity at index {entity_index} for document {document_id!r} in {filename}: {entity!r}"
            )

        try:
            start = int(start)
            end = int(end)
        except (TypeError, ValueError) as exc:
            raise TypeError(
                f"Non-integer span boundaries for document {document_id!r}, entity {entity_index}: {entity!r}"
            ) from exc
        if not isinstance(label, str):
            raise TypeError(f"Entity label must be a string, got {type(label).__name__}: {entity!r}")
        entities.append({"start": start, "end": end, "label": label, "text": entity_text})
    return entities


def _normalise_gold_map(raw_data, filename):
    if not isinstance(raw_data, dict):
        raise TypeError(f"{filename} should contain a JSON object keyed by document ID; got {type(raw_data).__name__}.")
    return {
        str(document_id): _normalise_entities_for_document(entities, str(document_id), filename)
        for document_id, entities in raw_data.items()
    }


with open(PHASE2_FILES["labels.txt"], "r", encoding="utf-8") as handle:
    labels_lines = [line.strip() for line in handle if line.strip() and not line.strip().startswith("#")]

# Accept either one label per line or a label followed by a human-readable description.
LABELS = []
for line in labels_lines:
    first_token = line.split()[0]
    if first_token in EXPECTED_PHASE2_LABELS:
        candidate = first_token
    elif line in EXPECTED_PHASE2_LABELS:
        candidate = line
    else:
        continue
    if candidate not in LABELS:
        LABELS.append(candidate)

if set(LABELS) != EXPECTED_PHASE2_LABELS:
    raise ValueError(
        "labels.txt did not resolve to the expected 12 Phase 2 labels. "
        f"Found={LABELS}; missing={sorted(EXPECTED_PHASE2_LABELS - set(LABELS))}; "
        f"unexpected={sorted(set(LABELS) - EXPECTED_PHASE2_LABELS)}"
    )

FOLD1_TEXTS = _normalise_text_map(_load_json_file("fold1-text.json"), "fold1-text.json")
FOLD1_GOLD = _normalise_gold_map(_load_json_file("fold1-gold.json"), "fold1-gold.json")
FOLD2_TEXTS = _normalise_text_map(_load_json_file("fold2-text.json"), "fold2-text.json")
FOLD2_GOLD = _normalise_gold_map(_load_json_file("fold2-gold.json"), "fold2-gold.json")
TEST_TEXTS = _normalise_text_map(_load_json_file("fold3-text.json"), "fold3-text.json")

for fold_name, texts, gold in (
    ("fold1", FOLD1_TEXTS, FOLD1_GOLD),
    ("fold2", FOLD2_TEXTS, FOLD2_GOLD),
):
    missing_gold = sorted(set(texts) - set(gold))
    orphan_gold = sorted(set(gold) - set(texts))
    if missing_gold or orphan_gold:
        raise ValueError(
            f"{fold_name} text/gold document IDs do not match. "
            f"Missing gold for {len(missing_gold)} IDs; gold without text for {len(orphan_gold)} IDs. "
            f"First missing={missing_gold[:5]}, first orphan={orphan_gold[:5]}"
        )

TEXTS_BY_ID = {}
ANNOTATIONS_BY_ID = {}
for fold_name, texts, gold in (
    ("fold1", FOLD1_TEXTS, FOLD1_GOLD),
    ("fold2", FOLD2_TEXTS, FOLD2_GOLD),
):
    overlap = set(TEXTS_BY_ID) & set(texts)
    if overlap:
        raise ValueError(
            f"Document IDs occur in both training folds ({len(overlap)} overlaps). "
            f"Example IDs: {sorted(overlap)[:10]}"
        )
    TEXTS_BY_ID.update(texts)
    ANNOTATIONS_BY_ID.update(gold)

TEST_OVERLAP = set(TEXTS_BY_ID) & set(TEST_TEXTS)
if TEST_OVERLAP:
    raise ValueError(
        f"Training and test document IDs overlap ({len(TEST_OVERLAP)} IDs). Example: {sorted(TEST_OVERLAP)[:10]}"
    )

# Strictly validate offsets and labels before tokenization; do not silently repair annotations.
ANNOTATION_COUNTS = Counter()
INVALID_SPANS = []
ENTITY_TEXT_MISMATCHES = []
for document_id, entities in ANNOTATIONS_BY_ID.items():
    source_text = TEXTS_BY_ID[document_id]
    for entity_index, entity in enumerate(entities):
        start, end, label = entity["start"], entity["end"], entity["label"]
        ANNOTATION_COUNTS[label] += 1
        if label not in EXPECTED_PHASE2_LABELS:
            INVALID_SPANS.append((document_id, entity_index, "unknown_label", label))
        if not (0 <= start < end <= len(source_text)):
            INVALID_SPANS.append((document_id, entity_index, "invalid_offsets", start, end, len(source_text)))
        elif entity["text"] is not None and source_text[start:end] != entity["text"]:
            ENTITY_TEXT_MISMATCHES.append((document_id, entity_index, entity["text"], source_text[start:end]))

if INVALID_SPANS:
    raise ValueError(
        f"Found {len(INVALID_SPANS)} invalid annotations. Examples: {INVALID_SPANS[:10]}"
    )

print("Official labels:", LABELS)
print(f"Fold 1: {len(FOLD1_TEXTS):,} documents, {sum(map(len, FOLD1_GOLD.values())):,} entities")
print(f"Fold 2: {len(FOLD2_TEXTS):,} documents, {sum(map(len, FOLD2_GOLD.values())):,} entities")
print(f"Combined training pool: {len(TEXTS_BY_ID):,} documents")
print(f"Fold 3 test documents: {len(TEST_TEXTS):,}")
print("Entity counts:")
for label in LABELS:
    print(f"  {label:10s} {ANNOTATION_COUNTS[label]:,}")
print("Optional entity_text mismatches (reported only, offsets preserved):", len(ENTITY_TEXT_MISMATCHES))
if ENTITY_TEXT_MISMATCHES:
    print("Mismatch examples:", ENTITY_TEXT_MISMATCHES[:3])

# Deterministic label-aware document-level holdout. Documents stay intact across splits.
# Labels that occur in only one document are kept in training when possible, rather than
# accidentally removing the only training example of a rare entity class.
all_document_ids = list(TEXTS_BY_ID)
if len(all_document_ids) < 2:
    raise ValueError("At least two labelled documents are required for a validation split.")

VALIDATION_FRACTION = 0.10
num_validation = max(1, int(round(len(all_document_ids) * VALIDATION_FRACTION)))
num_validation = min(num_validation, len(all_document_ids) - 1)
_doc_labels = {
    doc_id: {str(ent["label"]) for ent in ANNOTATIONS_BY_ID.get(doc_id, [])}
    for doc_id in all_document_ids
}
_label_doc_freq = Counter(label for doc_labels in _doc_labels.values() for label in doc_labels)
_target_val_counts = {
    label: (0 if freq <= 1 else min(freq - 1, max(1, int(round(freq * VALIDATION_FRACTION)))))
    for label, freq in _label_doc_freq.items()
}
_rng = random.Random(SEED)
_tie_break = {doc_id: _rng.random() for doc_id in all_document_ids}
_remaining = set(all_document_ids)
VAL_DOC_IDS = []
_val_label_counts = Counter()
while len(VAL_DOC_IDS) < num_validation:
    candidates = []
    for doc_id in _remaining:
        labels_here = _doc_labels[doc_id]
        # Preserve at least one labelled training document for every label occurring >=2 times.
        breaks_train_coverage = any(
            _label_doc_freq[label] >= 2 and
            sum(1 for other in _remaining if other != doc_id and label in _doc_labels[other]) == 0
            for label in labels_here
        )
        if breaks_train_coverage:
            continue
        gain = 0.0
        for label in labels_here:
            target = _target_val_counts.get(label, 0)
            freq = _label_doc_freq[label]
            if _val_label_counts[label] < target:
                gain += (target - _val_label_counts[label]) / max(1, freq) ** 0.5
            else:
                gain -= 0.15 / max(1, freq) ** 0.5
        # Fill validation with non-entity documents if needed, but prioritize label coverage.
        candidates.append((gain, -_tie_break[doc_id], doc_id))
    if not candidates:
        # A pathological label distribution can make the preservation constraint too strict.
        # Fall back only while keeping at least one training document overall.
        candidates = [(0.0, -_tie_break[doc_id], doc_id) for doc_id in _remaining]
    _, _, chosen = max(candidates)
    VAL_DOC_IDS.append(chosen)
    _remaining.remove(chosen)
    _val_label_counts.update(_doc_labels[chosen])

TRAIN_DOC_IDS = sorted(_remaining)
VAL_DOC_IDS = sorted(VAL_DOC_IDS)
print(f"Label-aware document split: train={len(TRAIN_DOC_IDS):,}, validation={len(VAL_DOC_IDS):,}")
print("Train/validation ID overlap:", len(set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS)))
assert not (set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS))
assert len(TRAIN_DOC_IDS) and len(VAL_DOC_IDS)
print("Train labels per class:", {l: sum(l in _doc_labels[d] for d in TRAIN_DOC_IDS) for l in LABELS})
print("Validation labels per class:", {l: sum(l in _doc_labels[d] for d in VAL_DOC_IDS) for l in LABELS})

## 5. Character offsets and BIO labels

BPE offsets are computed in Python-character coordinates. The character target construction does not snap entities to token boundaries; partial entities at window boundaries are ignored for that window, and overlapping/nested entities are explicitly masked because a single BIO sequence cannot represent them faithfully.

In [ ]:
import re


def _merge_two_offset_items(left_item, right_item, merged_token_id):
    spans = [
        (left_item[1], left_item[2]),
        (right_item[1], right_item[2]),
    ]
    covered = [(start, end) for start, end in spans if start is not None and end is not None]
    if not covered:
        return (merged_token_id, None, None)
    return (
        merged_token_id,
        min(start for start, _ in covered),
        max(end for _, end in covered),
    )


def _encode_word_with_offsets(word, word_start, tokenizer):
    items = []
    char_position = word_start
    for character in word:
        char_start = char_position
        char_end = char_position + len(character)
        for byte_value in character.encode("utf-8"):
            # This mirrors the existing ByteLevelBPE25k._encode_word implementation.
            items.append((4 + byte_value, char_start, char_end))
        char_position = char_end

    # The boundary token has no independent character span. If merged with word bytes,
    # the resulting token retains only the source characters occupied by those bytes.
    items.append((tokenizer.word_boundary_id, None, None))

    while len(items) > 1:
        best_position = None
        best_token_id = None
        for position in range(len(items) - 1):
            pair = (items[position][0], items[position + 1][0])
            merged_id = tokenizer.merge_to_id.get(pair)
            if merged_id is None:
                continue
            if best_token_id is None or merged_id < best_token_id:
                best_token_id = merged_id
                best_position = position
        if best_position is None:
            break
        position = best_position
        merged_item = _merge_two_offset_items(
            items[position], items[position + 1], best_token_id
        )
        items = items[:position] + [merged_item] + items[position + 2:]

    return items


def encode_with_offsets(text, tokenizer, add_bos=False, add_eos=False, verify_ids=True):
    """Return (token_ids, offsets), with offsets as Python-character [start, end) spans or None."""
    token_ids = []
    offsets = []

    if add_bos:
        token_ids.append(tokenizer.bos_token_id)
        offsets.append(None)

    matches = list(re.finditer(r"\S+", text, flags=re.UNICODE))
    regex_words = [match.group(0) for match in matches]
    if regex_words != text.split():
        raise RuntimeError("Whitespace matching differs from tokenizer.encode(text); refusing unsafe offsets.")

    for match in matches:
        word_items = _encode_word_with_offsets(match.group(0), match.start(), tokenizer)
        token_ids.extend(item[0] for item in word_items)
        offsets.extend(
            (item[1], item[2]) if item[1] is not None and item[2] is not None else None
            for item in word_items
        )

    if add_eos:
        token_ids.append(tokenizer.eos_token_id)
        offsets.append(None)

    if verify_ids:
        reference_ids = tokenizer.encode(text, add_bos=add_bos, add_eos=add_eos)
        if token_ids != reference_ids:
            raise AssertionError(
                "Offset-aware tokenization produced different token IDs from the original tokenizer."
            )
    if len(token_ids) != len(offsets):
        raise AssertionError("Token-ID/offset length mismatch.")
    return token_ids, offsets


OFFSET_TEST_STRINGS = [
    "The Supreme Court of India has jurisdiction over constitutional matters.",
    "  Case No. 123/2024\n\nCounsel for the appellant appeared.",
    "Unicode check: न्यायालय and café.",
    "tabs\tand multiple    spaces must preserve original character positions.",
]
for offset_test_text in OFFSET_TEST_STRINGS:
    test_ids, test_offsets = encode_with_offsets(offset_test_text, tokenizer)
    assert test_ids == tokenizer.encode(offset_test_text)
    assert len(test_ids) == len(test_offsets)
    for span in test_offsets:
        if span is not None:
            start, end = span
            assert 0 <= start < end <= len(offset_test_text)
print(f"Offset-aware tokenization checks passed for {len(OFFSET_TEST_STRINGS)} examples.")

In [ ]:
IGNORE_INDEX = -100
BIO_LABELS = ["O"] + [f"B-{label}" for label in LABELS] + [f"I-{label}" for label in LABELS]
BIO_TO_ID = {name: idx for idx, name in enumerate(BIO_LABELS)}
ID_TO_BIO = {idx: name for name, idx in BIO_TO_ID.items()}
O_LABEL_ID = BIO_TO_ID["O"]
print("BIO classes:", len(BIO_LABELS), "| O id:", O_LABEL_ID)

## 6. Exact-character targets, overlapping token windows, and dynamic padding

In [ ]:
# Character-level supervision, offset mapping, and dataset construction.
# Gold character offsets are used directly; no tokenizer-boundary approximation is used.

import os
import json
import copy
import bisect
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

CHAR_PAD_ID = 0
CHAR_UNK_ID = 1
MAX_CHAR_POSITION_IN_TOKEN = 32

# Build the character vocabulary from TRAIN documents only. Unseen validation/test
# characters map to CHAR_UNK_ID; validation/test text is not used to create it.
_training_chars = set()
for _doc_id in TRAIN_DOC_IDS:
    _training_chars.update(TEXTS_BY_ID[str(_doc_id)])
CHAR_TO_ID = {ch: i + 2 for i, ch in enumerate(sorted(_training_chars))}
CHAR_VOCAB_SIZE = len(CHAR_TO_ID) + 2


def _normalise_entity(entity):
    if isinstance(entity, dict):
        start = entity.get("start", entity.get("start_char", entity.get("start_offset")))
        end = entity.get("end", entity.get("end_char", entity.get("end_offset")))
        label = entity.get("label", entity.get("entity_label", entity.get("type")))
    elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
        start, end, label = entity[:3]
    else:
        raise TypeError(f"Unsupported entity representation: {entity!r}")
    start, end, label = int(start), int(end), str(label)
    if label not in LABELS:
        raise ValueError(f"Unknown entity label {label!r}")
    return start, end, label


def build_exact_character_targets(text, entities, char_start, char_end):
    """Return character BIO target IDs for [char_start, char_end) in the document.

    Fully contained gold entities receive exact B/I tags at original character
    positions. Entities cut by a window boundary are ignored in that window rather
    than taught as truncated entities. Overlapping/nested annotations are masked
    with IGNORE_INDEX because a single BIO sequence cannot represent nested spans faithfully.
    """
    if not (0 <= char_start <= char_end <= len(text)):
        raise ValueError("Invalid character window bounds.")
    length = char_end - char_start
    targets = [O_LABEL_ID] * length
    if length == 0:
        return targets

    parsed = []
    for entity in entities:
        s, e, label = _normalise_entity(entity)
        if not (0 <= s < e <= len(text)):
            raise ValueError(f"Invalid gold span {(s, e, label)} for text length {len(text)}")
        parsed.append((s, e, label))

    # Identify every annotation participating in an overlap/nested span.
    # Such regions are masked with IGNORE_INDEX rather than arbitrarily corrupted.
    conflict_ids = set()
    ordered = sorted(enumerate(parsed), key=lambda item: (item[1][0], item[1][1]))
    for pos, (idx_a, ent_a) in enumerate(ordered):
        s_a, e_a, _ = ent_a
        for idx_b, ent_b in ordered[pos + 1:]:
            s_b, e_b, _ = ent_b
            if s_b >= e_a:
                break
            if s_a < e_b and s_b < e_a:
                conflict_ids.add(idx_a)
                conflict_ids.add(idx_b)

    ignore_positions = set()
    for entity_idx, (s, e, label) in enumerate(parsed):
        overlap_start = max(s, char_start)
        overlap_end = min(e, char_end)
        if overlap_start >= overlap_end:
            continue

        # Only use complete entity examples for supervision in a window.
        is_partial = s < char_start or e > char_end
        if is_partial or entity_idx in conflict_ids:
            ignore_positions.update(range(overlap_start - char_start, overlap_end - char_start))
            continue

        b_id = BIO_TO_ID[f"B-{label}"]
        i_id = BIO_TO_ID[f"I-{label}"]
        local_start = s - char_start
        local_end = e - char_start
        targets[local_start] = b_id
        for p in range(local_start + 1, local_end):
            targets[p] = i_id

    for p in ignore_positions:
        targets[p] = IGNORE_INDEX
    return targets


def _map_characters_to_window_tokens(text, offsets, char_start, char_end):
    """Map each character to its local token index and relative token position.

    Whitespace and any other offset gaps are mapped to the nearest character-bearing
    token. Character targets remain exact; this mapping only supplies contextual
    token features to the character-level head.
    """
    length = char_end - char_start
    char_to_token = [-1] * length
    char_positions = [0] * length

    for token_idx, span in enumerate(offsets):
        if span is None or len(span) < 2 or span[0] is None or span[1] is None:
            continue
        token_start, token_end = int(span[0]), int(span[1])
        if token_start >= token_end:
            continue
        left = max(token_start, char_start)
        right = min(token_end, char_end)
        for absolute_char in range(left, right):
            local_char = absolute_char - char_start
            if 0 <= local_char < length and char_to_token[local_char] == -1:
                char_to_token[local_char] = token_idx
                char_positions[local_char] = min(absolute_char - token_start,
                                                  MAX_CHAR_POSITION_IN_TOKEN - 1)

    covered = [i for i, token_idx in enumerate(char_to_token) if token_idx >= 0]
    if not covered:
        return None, None

    for local_char in range(length):
        if char_to_token[local_char] >= 0:
            continue
        insertion = bisect.bisect_left(covered, local_char)
        left_char = covered[insertion - 1] if insertion > 0 else None
        right_char = covered[insertion] if insertion < len(covered) else None
        if left_char is None:
            source_char = right_char
        elif right_char is None:
            source_char = left_char
        elif local_char - left_char <= right_char - local_char:
            source_char = left_char
        else:
            source_char = right_char
        char_to_token[local_char] = char_to_token[source_char]
        char_positions[local_char] = char_positions[source_char]

    return char_to_token, char_positions


class CharacterBoundaryWindowDataset(Dataset):
    """Sliding token windows with exact character-level BIO supervision.
    Guarantees 100% character coverage across all documents.
    """
    def __init__(self, document_ids, text_by_id, entities_by_id, tokenizer_obj,
                 max_length=512, window_overlap=256):
        if not (0 <= window_overlap < max_length):
            raise ValueError("window_overlap must satisfy 0 <= overlap < max_length")
        self.samples = []
        self.document_ids = [str(x) for x in document_ids]
        step = max_length - window_overlap

        for doc_id in self.document_ids:
            text = text_by_id.get(doc_id, "")
            if not text:
                continue
            entities = entities_by_id.get(doc_id, [])
            token_ids, token_offsets = encode_with_offsets(text, tokenizer_obj)
            token_start = 0
            while token_start < len(token_ids):
                token_end = min(token_start + max_length, len(token_ids))
                window_ids = token_ids[token_start:token_end]
                window_offsets = token_offsets[token_start:token_end]
                valid_offsets = [
                    (int(span[0]), int(span[1]))
                    for span in window_offsets
                    if span is not None and span[0] is not None and span[1] is not None
                    and int(span[0]) < int(span[1])
                ]
                if valid_offsets:
                    char_start = min(s for s, _ in valid_offsets)
                    char_end = max(e for _, e in valid_offsets)

                    # Invariant: extend window 0 to start at char 0 and final window to len(text)
                    if token_start == 0:
                        char_start = 0
                    if token_end == len(token_ids):
                        char_end = len(text)

                    char_to_token, char_positions = _map_characters_to_window_tokens(
                        text, window_offsets, char_start, char_end
                    )
                    if char_to_token is not None:
                        char_ids = [CHAR_TO_ID.get(ch, CHAR_UNK_ID)
                                    for ch in text[char_start:char_end]]
                        char_labels = build_exact_character_targets(
                            text, entities, char_start, char_end
                        )
                        if not (len(char_ids) == len(char_to_token) == len(char_positions)
                                == len(char_labels)):
                            raise AssertionError("Character feature/target lengths differ")
                        if any(t < 0 or t >= len(window_ids) for t in char_to_token):
                            raise AssertionError("Character-to-token mapping is out of bounds")
                        self.samples.append({
                            "input_ids": torch.tensor(window_ids, dtype=torch.long),
                            "char_ids": torch.tensor(char_ids, dtype=torch.long),
                            "char_to_token": torch.tensor(char_to_token, dtype=torch.long),
                            "char_positions": torch.tensor(char_positions, dtype=torch.long),
                            "char_labels": torch.tensor(char_labels, dtype=torch.long),
                            "document_id": doc_id,
                            "token_start": token_start,
                            "char_start": char_start,
                            "char_end": char_end,
                        })
                if token_end == len(token_ids):
                    break
                token_start += step

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def audit_document_character_coverage(texts_by_id, dataset):
    """Audits that every character in each evaluation document is covered by at least one window."""
    coverage_by_doc = {str(doc_id): np.zeros(len(text), dtype=bool) for doc_id, text in texts_by_id.items()}
    for sample in dataset.samples:
        doc_id = sample["document_id"]
        if doc_id in coverage_by_doc:
            s = int(sample["char_start"])
            e = int(sample["char_end"])
            coverage_by_doc[doc_id][s:e] = True

    total_chars = sum(len(cov) for cov in coverage_by_doc.values())
    uncovered_chars = sum(np.sum(~cov) for cov in coverage_by_doc.values())
    coverage_rate = (total_chars - uncovered_chars) / max(total_chars, 1) * 100
    print(f"Document character coverage audit: {coverage_rate:.2f}% ({total_chars - uncovered_chars}/{total_chars} chars covered)")
    if uncovered_chars > 0:
        print(f"Warning: {uncovered_chars} characters were uncovered across {len(coverage_by_doc)} documents.")
    return coverage_by_doc, uncovered_chars == 0


def collate_character_windows(batch, pad_token_id=0):
    """Pads sequences dynamically within batch and returns aligned tensors."""
    if not batch:
        raise ValueError("Cannot collate an empty character-boundary batch")
    max_tokens = max(item["input_ids"].numel() for item in batch)
    max_chars = max(item["char_ids"].numel() for item in batch)
    batch_size = len(batch)

    input_ids = torch.full((batch_size, max_tokens), int(pad_token_id), dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_tokens), dtype=torch.float32)
    char_ids = torch.full((batch_size, max_chars), CHAR_PAD_ID, dtype=torch.long)
    char_to_token = torch.zeros((batch_size, max_chars), dtype=torch.long)
    char_positions = torch.zeros((batch_size, max_chars), dtype=torch.long)
    char_labels = torch.full((batch_size, max_chars), IGNORE_INDEX, dtype=torch.long)
    char_mask = torch.zeros((batch_size, max_chars), dtype=torch.float32)

    document_ids, char_starts, char_ends, char_lengths = [], [], [], []
    for row, item in enumerate(batch):
        token_len = item["input_ids"].numel()
        char_len = item["char_ids"].numel()
        input_ids[row, :token_len] = item["input_ids"]
        attention_mask[row, :token_len] = 1.0
        char_ids[row, :char_len] = item["char_ids"]
        char_to_token[row, :char_len] = item["char_to_token"]
        char_positions[row, :char_len] = item["char_positions"]
        char_labels[row, :char_len] = item["char_labels"]
        char_mask[row, :char_len] = 1.0
        document_ids.append(item["document_id"])
        char_starts.append(int(item["char_start"]))
        char_ends.append(int(item["char_end"]))
        char_lengths.append(int(char_len))

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "char_ids": char_ids,
        "char_to_token": char_to_token,
        "char_positions": char_positions,
        "char_labels": char_labels,
        "char_mask": char_mask,
        "document_ids": document_ids,
        "char_starts": char_starts,
        "char_ends": char_ends,
        "char_lengths": char_lengths,
    }

print("Character-level supervision, dataset, and collation utilities defined.")
print(f"Character vocabulary size: {CHAR_VOCAB_SIZE} (derived strictly from training split).")

## 7. Original Phase-1 model implementation copied from your uploaded `v4(1).ipynb`

The next cell uses the architecture code from the code cell that defines `HybridLegalLM` in your uploaded `v4(1).ipynb`. No model parameters/modules are added to the backbone. Only a `forward_hidden` helper is appended for the downstream character head.

**Important metadata finding:** your uploaded file is named `v4(1).ipynb`, but its own config labels the architecture `V2` / `lightweight_hybrid_decoder` and saves `hybrid_model.pt`. The notebook preserves this source exactly and does not infer model identity from a filename. The checkpoint loader accepts the compatible checkpoint format but reports the config recorded inside it.


In [ ]:
# 9. V2 Lightweight Hybrid Decoder-Only Transformer
#
# V2 preserves the successful V1 components:
# - RMSNorm
# - RoPE
# - Grouped Query Attention (GQA)
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache
#
# New architectural component:
# - Hybrid Global + Local Causal Attention
#
# Global layers:
#   Full causal attention over the complete sequence
#
# Local layers:
#   Causal attention restricted to a fixed local window
#
# The hybrid design is motivated by the course discussion of
# sparse/local attention and hybrid models that interleave
# full and sparse attention layers.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Hybrid Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        attention_type="global",
        local_window=256,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        assert attention_type in {
            "global",
            "local"
        }

        assert local_window > 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.attention_type = attention_type
        self.local_window = local_window

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def _build_attention_mask(
        self,
        query_length,
        key_length,
        query_position_offset,
        device
    ):
        """
        Build a boolean attention mask.

        True  = attention is allowed
        False = attention is blocked

        For global attention:
            causal mask

        For local attention:
            causal + fixed sliding window
        """

        query_positions = (
            query_position_offset
            + torch.arange(
                query_length,
                device=device
            )
        )

        key_positions = torch.arange(
            key_length,
            device=device
        )

        # Causal condition:
        # query at position q can only see keys <= q.
        causal_mask = (
            key_positions.unsqueeze(0)
            <= query_positions.unsqueeze(1)
        )

        if self.attention_type == "global":
            return causal_mask

        # Local causal attention:
        # q can see at most the previous
        # local_window - 1 tokens plus itself.
        local_mask = (
            key_positions.unsqueeze(0)
            >= (
                query_positions.unsqueeze(1)
                - self.local_window
                + 1
            )
        )

        return causal_mask & local_mask

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        # -------------------------------------------------
        # Q, K, V projections
        # -------------------------------------------------

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        # -------------------------------------------------
        # KV cache information
        # -------------------------------------------------

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # -------------------------------------------------
        # RoPE
        # -------------------------------------------------

        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # -------------------------------------------------
        # Append new K/V to cache
        # -------------------------------------------------

        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # -------------------------------------------------
        # GQA
        # -------------------------------------------------

        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # -------------------------------------------------
        # Hybrid attention mask
        # -------------------------------------------------

        key_length = k.size(-2)

        attn_mask = self._build_attention_mask(
            query_length=T,
            key_length=key_length,
            query_position_offset=past_len,
            device=x.device
        )

        # -------------------------------------------------
        # Scaled dot-product attention
        # -------------------------------------------------

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        # -------------------------------------------------
        # Output projection
        # -------------------------------------------------

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard GELU FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Hybrid Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout,
        layer_index,
        global_attention_layers,
        local_window
    ):
        super().__init__()

        self.layer_index = layer_index

        if layer_index in global_attention_layers:
            attention_type = "global"
        else:
            attention_type = "local"

        self.attention_type = attention_type

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            attention_type=attention_type,
            local_window=local_window,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention + residual.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN + residual.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V2 Lightweight Hybrid Legal Language Model
# ---------------------------------------------------------

class HybridLegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout,
                layer_index=layer_index,
                global_attention_layers=GLOBAL_ATTENTION_LAYERS,
                local_window=LOCAL_ATTN_WINDOW
            )
            for layer_index in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Weight tying retained from V1.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

# Phase-2-only helper: reuses the original V1/V2 hybrid backbone blocks exactly and
# returns contextual token states without computing the vocabulary projection.
# This adds no parameters and does not change the pretrained state-dictionary keys.
def _hybrid_forward_hidden(self, input_ids):
    x = self.token_embedding(input_ids)
    for block in self.blocks:
        x, _ = block(x, past_key_value=None, use_cache=False)
    return self.final_norm(x)

HybridLegalLM.forward_hidden = _hybrid_forward_hidden


## 8. Restore the uploaded Phase-1 hybrid checkpoint strictly

A model checkpoint is not self-describing by itself; this step validates its config and loads every pretrained tensor with `strict=True`. If this cell fails, stop and check that the attached checkpoint and tokenizer belong to uploaded Phase-1 hybrid checkpoint.

In [ ]:
# Restore the checkpoint saved by the exact uploaded v4(1).ipynb (usually hybrid_model.pt); alternatively, v4_model.pt is accepted only if it is the compatible intended checkpoint.
# The class implementation above is copied verbatim from the architecture cell in the uploaded v4(1).ipynb (HybridLegalLM).
if HYBRID_CHECKPOINT_PATH:
    if not os.path.isfile(HYBRID_CHECKPOINT_PATH):
        raise FileNotFoundError(f"HYBRID_CHECKPOINT_PATH does not exist: {HYBRID_CHECKPOINT_PATH}")
else:
    checkpoint_candidates = []
    accepted_names = {"hybrid_model.pt", "v4_model.pt"}
    for search_root in ("/kaggle/input", "/kaggle/working"):
        if os.path.isdir(search_root):
            for root, _, files in os.walk(search_root):
                for filename in sorted(accepted_names.intersection(files)):
                    checkpoint_candidates.append(os.path.join(root, filename))
    checkpoint_candidates = sorted(set(checkpoint_candidates))
    if len(checkpoint_candidates) == 1:
        HYBRID_CHECKPOINT_PATH = checkpoint_candidates[0]
    elif not checkpoint_candidates:
        raise FileNotFoundError(
            "No hybrid_model.pt or v4_model.pt found. Attach the Phase-1 checkpoint dataset "
            "and set HYBRID_CHECKPOINT_PATH explicitly."
        )
    else:
        raise RuntimeError(
            "Multiple candidate checkpoints found. Set HYBRID_CHECKPOINT_PATH explicitly to the intended file:\n"
            + "\n".join(checkpoint_candidates)
        )

try:
    phase1_checkpoint = torch.load(HYBRID_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
except TypeError:  # compatibility with older PyTorch
    phase1_checkpoint = torch.load(HYBRID_CHECKPOINT_PATH, map_location="cpu")
if not isinstance(phase1_checkpoint, dict) or "model_state_dict" not in phase1_checkpoint:
    raise KeyError("Expected a Phase-1 checkpoint dict containing 'model_state_dict'.")
config = phase1_checkpoint.get("config", {})
if not isinstance(config, dict):
    raise TypeError("Checkpoint config must be a dictionary.")

HYBRID_CONFIG = config
CHECKPOINT_ARCHITECTURE_LABEL = str(config.get("architecture_version", config.get("architecture", "not recorded")))
HYBRID_VOCAB_SIZE = int(config.get("vocab_size", tokenizer.vocab_size))
N_EMBD = int(config.get("n_embd", 768))
N_LAYER = int(config.get("n_layer", 12))
N_HEAD = int(config.get("n_head", 12))
N_KV_HEAD = int(config.get("n_kv_head", 4))
FFN_MULTIPLIER = int(config.get("ffn_multiplier", 4))
DROPOUT = float(config.get("dropout", 0.0))
MODEL_CONTEXT_LENGTH = int(config.get("context_length", config.get("max_seq_len", 1024)))
# The uploaded v4(1).ipynb stores local_attention_window/global_attention_layers in its config.
LOCAL_WINDOW_SIZE = int(config.get("local_attention_window", config.get("local_window_size", 256)))
GLOBAL_LAYERS = tuple(int(x) for x in config.get("global_attention_layers", config.get("global_layers", [0, 4, 8, 11])))
ATTENTION_MODE = str(config.get("attention_mode", "hybrid"))

if HYBRID_VOCAB_SIZE != tokenizer.vocab_size:
    raise ValueError(f"Tokenizer/checkpoint vocab mismatch: tokenizer={tokenizer.vocab_size}, checkpoint={HYBRID_VOCAB_SIZE}")
if N_LAYER < 6:
    raise ValueError(f"Checkpoint has only {N_LAYER} layers; the planned six-layer student requires at least 6.")
if N_EMBD % N_HEAD != 0 or N_HEAD % N_KV_HEAD != 0:
    raise ValueError("Invalid attention dimension configuration in checkpoint metadata.")
if LOCAL_WINDOW_SIZE < 1 or LOCAL_WINDOW_SIZE > MODEL_CONTEXT_LENGTH:
    raise ValueError(f"Invalid local window {LOCAL_WINDOW_SIZE} for context {MODEL_CONTEXT_LENGTH}.")
if not GLOBAL_LAYERS or any(x < 0 or x >= N_LAYER for x in GLOBAL_LAYERS):
    raise ValueError(f"Invalid global-attention layer list in checkpoint config: {GLOBAL_LAYERS}")
if ATTENTION_MODE not in {"hybrid", "global_local", "lightweight_hybrid_decoder", ""}:
    warnings.warn(f"Unexpected attention_mode metadata {ATTENTION_MODE!r}; verify this is the intended hybrid checkpoint.")
MAX_TOKEN_LENGTH = min(MAX_TOKEN_LENGTH, MODEL_CONTEXT_LENGTH)

# The supplied class constructs layer types from these globals. Set them from the checkpoint BEFORE instantiation.
GLOBAL_ATTENTION_LAYERS = list(GLOBAL_LAYERS)
LOCAL_ATTN_WINDOW = LOCAL_WINDOW_SIZE

# Preserve original pretrained weights for the student layer mapping.
phase1_state = {k: v.detach().cpu().clone() for k, v in phase1_checkpoint["model_state_dict"].items()}

hybrid_backbone = HybridLegalLM(
    vocab_size=HYBRID_VOCAB_SIZE,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    max_seq_len=MODEL_CONTEXT_LENGTH,
    ffn_multiplier=FFN_MULTIPLIER,
    dropout=DROPOUT,
)
load_result = hybrid_backbone.load_state_dict(phase1_state, strict=True)

print("Strictly loaded Phase-1 hybrid checkpoint:", HYBRID_CHECKPOINT_PATH)
print("Checkpoint architecture metadata:", CHECKPOINT_ARCHITECTURE_LABEL)
if str(config.get("architecture_version", "")).upper() == "V2" or "V2" in str(config.get("architecture", "")).upper():
    print("NOTE: the checkpoint metadata says V2. The filename alone cannot override this internal metadata. This experiment intentionally uses the exact submitted file without relabeling its architecture.")
    print("      The supplied model is a decoder-only hybrid global/local attention network; verify the checkpoint you attach is the intended best-performing Phase-1 hybrid model before training.")
print("Strict load result:", load_result)
print("Model config:", {"layers": N_LAYER, "hidden": N_EMBD, "heads": N_HEAD,
                        "kv_heads": N_KV_HEAD, "context": MODEL_CONTEXT_LENGTH,
                        "local_window": LOCAL_WINDOW_SIZE, "global_layers": GLOBAL_LAYERS,
                        "vocab_size": HYBRID_VOCAB_SIZE})
print(f"Pretrained parameters: {sum(p.numel() for p in hybrid_backbone.parameters()):,}")


## 9. LoRMA: multiplicative low-rank adaptation on hybrid Q/V projections

This is a multiplicative update of each pretrained projection output, not ordinary additive LoRA. We first load the checkpoint, then wrap the frozen Q and V linear projections. Zero-initializing the second low-rank matrix makes insertion function-preserving at step zero.

### Device-placement fix

The adapter modules must be created on the same device and dtype as the pretrained projection they wrap. The previous implementation created `lorma_down` and `lorma_up` on CPU even when the hybrid backbone had already been moved to CUDA, causing `mat2 is on cpu, different from other tensor on cuda:0`. The implementation below inherits the base projection's device and dtype and asserts placement during injection.


In [ ]:
class LoRMALinearPlus(nn.Module):
    """LoRMA-style multiplicative adapter: y'=(I + alpha/r * B A) (W x).

    This is not additive LoRA. The pretrained linear projection is kept intact and
    the low-rank transform acts multiplicatively on its output. Zero-initializing B
    makes the adapter an exact identity at initialization.
    """
    def __init__(self, base_layer, rank=8, alpha=16.0, dropout=0.05):
        super().__init__()
        if not isinstance(base_layer, nn.Linear):
            raise TypeError("LoRMALinearPlus can wrap only nn.Linear layers")
        if not (1 <= rank <= base_layer.out_features):
            raise ValueError(f"Invalid LoRMA rank={rank} for out_features={base_layer.out_features}")
        self.base_layer = base_layer
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scale = self.alpha / self.rank
        self.adapter_dropout = nn.Dropout(float(dropout))
        # Important: adapter layers are injected after the Phase-1 backbone may
        # already reside on CUDA. New nn.Linear modules default to CPU unless
        # device/dtype are specified, which caused the reported CPU/CUDA mismatch.
        adapter_device = base_layer.weight.device
        adapter_dtype = base_layer.weight.dtype
        self.lorma_down = nn.Linear(
            base_layer.out_features, self.rank, bias=False,
            device=adapter_device, dtype=adapter_dtype,
        )
        self.lorma_up = nn.Linear(
            self.rank, base_layer.out_features, bias=False,
            device=adapter_device, dtype=adapter_dtype,
        )
        nn.init.kaiming_uniform_(self.lorma_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_up.weight)
        if (self.lorma_down.weight.device != base_layer.weight.device or
                self.lorma_up.weight.device != base_layer.weight.device):
            raise RuntimeError(
                "LoRMA adapter device mismatch: base=" + str(base_layer.weight.device) +
                ", down=" + str(self.lorma_down.weight.device) +
                ", up=" + str(self.lorma_up.weight.device)
            )
        for parameter in self.base_layer.parameters():
            parameter.requires_grad = False

    def forward(self, x):
        base_output = self.base_layer(x)
        multiplicative_delta = self.lorma_up(self.lorma_down(self.adapter_dropout(base_output)))
        return base_output + self.scale * multiplicative_delta


def apply_lorma_to_hybrid(backbone, rank=LORMA_RANK, alpha=LORMA_ALPHA, dropout=LORMA_DROPOUT):
    """Apply LoRMA to q/v projections AFTER the original checkpoint has loaded strictly."""
    for parameter in backbone.parameters():
        parameter.requires_grad = False
    wrapped = 0
    for block in backbone.blocks:
        attention = block.attention
        for attr in ("q_proj", "v_proj"):
            base = getattr(attention, attr)
            if isinstance(base, LoRMALinearPlus):
                raise RuntimeError(f"{attr} already has a LoRMA wrapper")
            wrapped_layer = LoRMALinearPlus(base, rank=rank, alpha=alpha, dropout=dropout)
            if (wrapped_layer.lorma_down.weight.device != base.weight.device or
                    wrapped_layer.lorma_up.weight.device != base.weight.device):
                raise RuntimeError(
                    f"LoRMA device mismatch at block={block.layer_index}, projection={attr}: "
                    f"base={base.weight.device}, down={wrapped_layer.lorma_down.weight.device}, "
                    f"up={wrapped_layer.lorma_up.weight.device}"
                )
            setattr(attention, attr, wrapped_layer)
            wrapped += 1
    if wrapped != 2 * len(backbone.blocks):
        raise RuntimeError("Not all hybrid q/v projections received a LoRMA adapter")
    return backbone

# Check exact identity initialization on actual loaded hybrid weights before training.
_v4_probe = torch.randint(0, HYBRID_VOCAB_SIZE, (1, min(12, MAX_TOKEN_LENGTH)), device=DEVICE)
hybrid_backbone = hybrid_backbone.to(DEVICE).eval()
with torch.no_grad():
    _probe_logits_before, _ = hybrid_backbone(_v4_probe, use_cache=False)
apply_lorma_to_hybrid(hybrid_backbone)
with torch.no_grad():
    _probe_logits_after, _ = hybrid_backbone(_v4_probe, use_cache=False)
_lorma_identity_error = (_probe_logits_before - _probe_logits_after).abs().max().item()
print(f"LoRMA identity-init max logit difference: {_lorma_identity_error:.8f}")
assert _lorma_identity_error < 1e-5, "LoRMA insertion altered the hybrid function at initialization."

_adapter_devices = sorted({str(p.device) for n, p in hybrid_backbone.named_parameters() if "lorma_down" in n or "lorma_up" in n})
_base_projection_devices = sorted({str(p.device) for n, p in hybrid_backbone.named_parameters() if "base_layer.weight" in n})
print("LoRMA adapter devices:", _adapter_devices, "| base projection devices:", _base_projection_devices)
if _adapter_devices != _base_projection_devices:
    raise RuntimeError(f"Injected LoRMA adapters and base projections are on different devices: {_adapter_devices} vs {_base_projection_devices}")

## 10. Character-boundary head and LoRMA teacher

In [ ]:
class CharacterBoundaryHead(nn.Module):
    """Emit BIO logits for each raw-text character using contextual token features.
    Substantive modelling improvements:
    - Multi-scale dilated convolutions (local sub-word context + full-word dilated context)
    - Channel LayerNorm for training stability
    - Residual connections across convolutional stages
    """
    def __init__(self, hidden_dim, char_vocab_size, num_labels, char_dim=32,
                 token_feature_dim=128, relative_position_dim=16, dropout=0.1):
        super().__init__()
        self.token_projection = nn.Linear(hidden_dim, token_feature_dim)
        self.char_embedding = nn.Embedding(
            char_vocab_size, char_dim, padding_idx=CHAR_PAD_ID
        )
        self.relative_position_embedding = nn.Embedding(
            MAX_CHAR_POSITION_IN_TOKEN, relative_position_dim
        )
        joined_dim = token_feature_dim + char_dim + relative_position_dim
        self.input_projection = nn.Sequential(
            nn.Linear(joined_dim, token_feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.norm0 = nn.LayerNorm(token_feature_dim)

        # Conv1: receptive field 5 (local char context: sub-word/affixes)
        self.conv1 = nn.Conv1d(token_feature_dim, token_feature_dim, kernel_size=5, padding=2)
        self.norm1 = nn.LayerNorm(token_feature_dim)

        # Conv2: dilated conv with dilation 2 (receptive field spans ~13 chars: full word context)
        self.conv2 = nn.Conv1d(token_feature_dim, token_feature_dim, kernel_size=5, padding=4, dilation=2)
        self.norm2 = nn.LayerNorm(token_feature_dim)

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(token_feature_dim, num_labels)

    def forward(self, token_hidden_states, char_ids, char_to_token,
                char_positions, char_mask):
        batch_size, char_length = char_ids.shape
        safe_map = char_to_token.clamp(min=0, max=token_hidden_states.shape[1] - 1)
        gather_index = safe_map.unsqueeze(-1).expand(-1, -1, token_hidden_states.shape[-1])
        char_context = token_hidden_states.gather(dim=1, index=gather_index)

        features = torch.cat([
            self.token_projection(char_context),
            self.char_embedding(char_ids),
            self.relative_position_embedding(
                char_positions.clamp(min=0, max=MAX_CHAR_POSITION_IN_TOKEN - 1)
            ),
        ], dim=-1)
        x = self.input_projection(features)
        mask = char_mask.unsqueeze(-1)
        x = self.norm0(x) * mask

        # Conv1 with residual
        conv1_out = self.conv1(x.transpose(1, 2)).transpose(1, 2)
        h1 = self.norm1(F.gelu(conv1_out) + x) * mask
        h1 = self.dropout(h1)

        # Conv2 with residual
        conv2_out = self.conv2(h1.transpose(1, 2)).transpose(1, 2)
        h2 = self.norm2(F.gelu(conv2_out) + h1) * mask
        h2 = self.dropout(h2)

        return self.classifier(h2)

class HybridCharacterNERModel(nn.Module):
    """Uploaded pretrained hybrid decoder-only backbone plus exact-character BIO head."""
    def __init__(self, backbone, char_vocab_size, num_labels):
        super().__init__()
        self.backbone = backbone
        self.char_head = CharacterBoundaryHead(
            hidden_dim=N_EMBD,
            char_vocab_size=char_vocab_size,
            num_labels=num_labels,
            dropout=0.1,
        )

    def forward(self, input_ids, attention_mask, char_ids, char_to_token,
                char_positions, char_mask):
        # Batches are right-padded. Causal attention prevents valid positions
        # from seeing trailing PAD tokens; only valid token positions are gathered.
        if input_ids.shape != attention_mask.shape:
            raise ValueError("input_ids and attention_mask shapes differ")
        token_hidden = self.backbone.forward_hidden(input_ids)
        return self.char_head(token_hidden, char_ids, char_to_token,
                              char_positions, char_mask)

# The pretrained backbone is frozen; only LoRMA adapter matrices and the character head train.
for parameter in HybridCharacterNERModel.__dict__.get("_never_used", []):
    parameter.requires_grad = False
teacher_model = HybridCharacterNERModel(
    backbone=hybrid_backbone, char_vocab_size=CHAR_VOCAB_SIZE, num_labels=len(BIO_LABELS)
).to(DEVICE)
for name, parameter in teacher_model.named_parameters():
    parameter.requires_grad = ("lorma_down" in name or "lorma_up" in name or "char_head" in name)

teacher_trainable = [(n,p) for n,p in teacher_model.named_parameters() if p.requires_grad]
teacher_total = sum(p.numel() for p in teacher_model.parameters())
teacher_trainable_count = sum(p.numel() for _,p in teacher_trainable)
print(f"Hybrid LoRMA teacher parameters: {teacher_total:,}")
print(f"Trainable LoRMA + character-head parameters: {teacher_trainable_count:,} ({100*teacher_trainable_count/max(teacher_total,1):.3f}%)")
assert teacher_trainable_count > 0
assert all(("lorma_down" in n or "lorma_up" in n or "char_head" in n) for n,p in teacher_trainable)

## 11. Preflight: actual checkpoint, adapter insertion, and character-head forward path

In [ ]:
# Real-checkpoint forward/backward preflight without an optimizer step.
teacher_model.eval()
_smoke_tokens = torch.randint(0, HYBRID_VOCAB_SIZE, (1, min(24, MAX_TOKEN_LENGTH)), device=DEVICE)
_smoke_hidden = teacher_model.backbone.forward_hidden(_smoke_tokens)
assert _smoke_hidden.shape == (1, _smoke_tokens.shape[1], N_EMBD)
_smoke_char_len = min(40, _smoke_tokens.shape[1] * 4)
_smoke_inputs = {
    "input_ids": _smoke_tokens,
    "attention_mask": torch.ones_like(_smoke_tokens, dtype=torch.float32),
    "char_ids": torch.randint(0, CHAR_VOCAB_SIZE, (1, _smoke_char_len), device=DEVICE),
    "char_to_token": torch.randint(0, _smoke_tokens.shape[1], (1, _smoke_char_len), device=DEVICE),
    "char_positions": torch.randint(0, MAX_CHAR_POSITION_IN_TOKEN, (1, _smoke_char_len), device=DEVICE),
    "char_mask": torch.ones((1, _smoke_char_len), device=DEVICE),
}
with torch.no_grad():
    _smoke_logits = teacher_model(**_smoke_inputs)
assert _smoke_logits.shape == (1, _smoke_char_len, len(BIO_LABELS))
assert torch.isfinite(_smoke_logits).all()
print("Uploaded hybrid checkpoint + LoRMA + character head preflight passed:", tuple(_smoke_logits.shape))
print("Note: This is a small smoke test, not a training run or evidence of leaderboard performance.")

## 12. Strict span metrics, overlap aggregation, loss, and submission helpers

All 12 entity classes contribute to the official-style strict Macro-F1. Class weights are calculated after deduplicating absolute document characters across overlapping windows. The decoder retains one-character spans and rejects invalid/duplicate output later.

In [ ]:
def strict_span_metrics(gold_by_id, predicted_by_id, labels, document_ids):
    per_label = {}
    f1s = []
    for label in labels:
        gold_set, pred_set = set(), set()
        for doc_id in document_ids:
            for ent in gold_by_id.get(str(doc_id), []):
                if str(ent["label"]) == label:
                    gold_set.add((str(doc_id), int(ent["start"]), int(ent["end"]), label))
            for ent in predicted_by_id.get(str(doc_id), []):
                if str(ent[2]) == label:
                    pred_set.add((str(doc_id), int(ent[0]), int(ent[1]), label))
        tp = len(gold_set & pred_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[label] = {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        f1s.append(f1)
    return {
        "strict_macro_f1": sum(f1s) / max(1, len(f1s)),
        "macro_precision": sum(v["precision"] for v in per_label.values()) / max(1, len(labels)),
        "macro_recall": sum(v["recall"] for v in per_label.values()) / max(1, len(labels)),
        "total_true_positives": sum(v["tp"] for v in per_label.values()),
        "total_false_positives": sum(v["fp"] for v in per_label.values()),
        "total_false_negatives": sum(v["fn"] for v in per_label.values()),
        "per_label": per_label,
    }


def _character_model_inputs(batch, device):
    return {key: batch[key].to(device) for key in
            ("input_ids", "attention_mask", "char_ids", "char_to_token", "char_positions", "char_mask")}


def collect_document_character_logits(model, dataset, texts_by_id, batch_size=2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0,
                        collate_fn=lambda items: collate_character_windows(items, PAD_TOKEN_ID))
    model.eval()
    accumulated, accumulated_weight = {}, {}
    with torch.inference_mode():
        for batch in loader:
            inputs = _character_model_inputs(batch, DEVICE)
            with (torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else __import__('contextlib').nullcontext()):
                logits = model(**inputs)
            logits_cpu = logits.float().cpu().numpy()
            for row, doc_id in enumerate(batch["document_ids"]):
                doc_id = str(doc_id)
                text = texts_by_id[doc_id]
                if doc_id not in accumulated:
                    accumulated[doc_id] = np.zeros((len(text), len(BIO_LABELS)), dtype=np.float32)
                    accumulated_weight[doc_id] = np.zeros(len(text), dtype=np.float32)
                start = int(batch["char_starts"][row])
                length = int(batch["char_lengths"][row])
                if length <= 0:
                    continue
                end = min(start + length, len(text))
                length = end - start
                positions = np.arange(length)
                edge = np.minimum(positions, length - 1 - positions)
                weights = 1.0 + np.minimum(edge, 64).astype(np.float32) / 64.0
                accumulated[doc_id][start:end] += logits_cpu[row, :length, :] * weights[:, None]
                accumulated_weight[doc_id][start:end] += weights
    result = {}
    for raw_doc_id, text in texts_by_id.items():
        doc_id = str(raw_doc_id)
        if len(text) == 0:
            result[doc_id] = np.zeros((0, len(BIO_LABELS)), dtype=np.float32)
        elif doc_id not in accumulated:
            fallback = np.full((len(text), len(BIO_LABELS)), -5.0, dtype=np.float32)
            fallback[:, O_LABEL_ID] = 0.0
            result[doc_id] = fallback
        else:
            weights = accumulated_weight[doc_id]
            covered = weights > 0
            out = np.full((len(text), len(BIO_LABELS)), -5.0, dtype=np.float32)
            out[:, O_LABEL_ID] = 0.0
            out[covered] = accumulated[doc_id][covered] / weights[covered, None]
            result[doc_id] = out
    return result


def decode_character_bio_logits(logits, thresholds=None, min_entity_chars=1):
    arr = np.asarray(logits, dtype=np.float32).copy()
    if thresholds:
        for label, bias in thresholds.items():
            arr[:, BIO_TO_ID[f"B-{label}"]] -= float(bias)
            arr[:, BIO_TO_ID[f"I-{label}"]] -= float(bias)
    tag_ids = arr.argmax(axis=-1).tolist()
    spans, active_label, active_start = [], None, None
    def flush(end):
        nonlocal active_label, active_start
        if active_label is not None and active_start is not None and end - active_start >= min_entity_chars:
            spans.append([int(active_start), int(end), str(active_label)])
        active_label, active_start = None, None
    for pos, tag_id in enumerate(tag_ids):
        tag = ID_TO_BIO[int(tag_id)]
        if tag.startswith("B-"):
            flush(pos)
            active_label, active_start = tag[2:], pos
        elif tag.startswith("I-"):
            label = tag[2:]
            if active_label != label:
                flush(pos)  # Strict BIO decoding: orphan I-tags are not promoted to B-tags.
        else:
            flush(pos)
    flush(len(tag_ids))
    return spans


def score_document_logits(logits_by_doc, texts_by_id, gold_by_id, thresholds=None):
    predictions = {str(doc_id): decode_character_bio_logits(logits, thresholds=thresholds)
                   for doc_id, logits in logits_by_doc.items()}
    doc_ids = [str(x) for x in texts_by_id]
    for doc_id in doc_ids:
        predictions.setdefault(doc_id, [])
    metrics = strict_span_metrics({str(k): v for k,v in gold_by_id.items()}, predictions, LABELS, doc_ids)
    metrics["predictions_by_doc"] = predictions
    return metrics


def unique_character_class_weights(dataset, device):
    """Count each supervised document character once, even when windows overlap."""
    labels_by_char = {}
    for sample in dataset.samples:
        doc_id = str(sample["document_id"])
        start = int(sample["char_start"])
        for rel, tag_id in enumerate(sample["char_labels"].tolist()):
            tag_id = int(tag_id)
            if tag_id == IGNORE_INDEX:
                continue
            key = (doc_id, start + rel)
            previous = labels_by_char.get(key)
            if previous is not None and previous != tag_id:
                raise ValueError(f"Conflicting non-ignored labels for absolute character {key}: {previous} vs {tag_id}")
            labels_by_char[key] = tag_id
    if not labels_by_char:
        raise ValueError("No supervised characters found for loss-weight calculation.")
    counts = torch.bincount(torch.tensor(list(labels_by_char.values()), dtype=torch.long), minlength=len(BIO_LABELS)).float().clamp_min(1.0)
    weights = (counts[O_LABEL_ID] / counts).pow(0.20).clamp(1.0, 2.5)
    weights[O_LABEL_ID] = 1.0
    print(f"Unique supervised characters={len(labels_by_char):,}; class-weight range={weights.min().item():.3f}..{weights.max().item():.3f}")
    return weights.to(device)


class WeightedFocalLoss(nn.Module):
    def __init__(self, class_weights, gamma=1.0, ignore_index=IGNORE_INDEX):
        super().__init__()
        self.register_buffer("class_weights", class_weights.detach().float().clone())
        self.gamma = float(gamma)
        self.ignore_index = int(ignore_index)
    def forward(self, logits, targets):
        logits = logits.reshape(-1, logits.shape[-1])
        targets = targets.reshape(-1).long()
        valid = targets.ne(self.ignore_index)
        if not valid.any():
            return logits.sum() * 0.0
        logp = F.log_softmax(logits[valid].float(), dim=-1)
        y = targets[valid]
        log_pt = logp.gather(1, y[:, None]).squeeze(1)
        alpha = self.class_weights[y]
        loss = -alpha * (1.0 - log_pt.exp()).clamp(0, 1).pow(self.gamma) * log_pt
        return loss.sum() / alpha.sum().clamp_min(1e-8)


def temperature_scaled_kd_loss(student_logits, teacher_logits, valid_mask, temperature=2.0):
    """Float32 temperature-scaled KL loss over valid character positions only.

    valid_mask must already exclude padding and IGNORE_INDEX target positions.
    Teacher logits are detached so gradients flow only into the student.
    """
    if student_logits.shape != teacher_logits.shape:
        raise ValueError(f"KD logit shape mismatch: student={tuple(student_logits.shape)}, teacher={tuple(teacher_logits.shape)}")
    if valid_mask.shape != student_logits.shape[:-1]:
        raise ValueError(f"KD mask shape mismatch: mask={tuple(valid_mask.shape)}, logits={tuple(student_logits.shape)}")
    if temperature <= 0:
        raise ValueError("KD temperature must be positive")
    student_log_probs = F.log_softmax(student_logits.float() / float(temperature), dim=-1)
    teacher_probs = F.softmax(teacher_logits.detach().float() / float(temperature), dim=-1)
    per_position = F.kl_div(student_log_probs, teacher_probs, reduction="none").sum(dim=-1)
    mask = valid_mask.bool()
    if not mask.any():
        return student_logits.float().sum() * 0.0
    return (per_position * mask.float()).sum() / mask.float().sum().clamp_min(1.0) * (float(temperature) ** 2)

def evaluate_character_model(model, dataset, texts, gold, batch_size=BATCH_SIZE):
    logits = collect_document_character_logits(model, dataset, texts, batch_size=batch_size)
    metrics = score_document_logits(logits, texts, gold, thresholds=None)
    metrics["document_logits"] = logits
    return metrics

In [ ]:
# Fast synthetic regression tests. These do not need GPU training or the private test labels.
# Keep these tests green before launching the expensive hybrid fine-tuning/distillation runs.
import numpy as np

def run_phase2_synthetic_regression_tests():
    # Exact BIO target on a one-character span.
    target = build_exact_character_targets("A B", [{"start": 0, "end": 1, "label": "APP"}], 0, 3)
    assert target[0] == BIO_TO_ID["B-APP"], target
    assert target[1] == O_LABEL_ID, target

    # A window-cut entity must be ignored rather than learned as a truncated mention.
    partial = build_exact_character_targets("ABCD", [{"start": 0, "end": 3, "label": "APP"}], 1, 4)
    assert partial[0] == IGNORE_INDEX, partial

    # Overlapping/nested entities cannot be represented in one BIO sequence: mask both.
    overlapping = build_exact_character_targets("ABCD", [
        {"start": 0, "end": 3, "label": "APP"},
        {"start": 1, "end": 2, "label": "RESP"},
    ], 0, 4)
    assert all(v == IGNORE_INDEX for v in overlapping[:3]), overlapping

    # One-character exact span decode and strict span metric must distinguish a shift.
    logits = np.full((1, len(BIO_LABELS)), -5.0, dtype=np.float32)
    logits[0, O_LABEL_ID] = 0.0
    logits[0, BIO_TO_ID["B-APP"]] = 5.0
    assert decode_character_bio_logits(logits) == [[0, 1, "APP"]]
    gold_test = {"synthetic": [{"start": 0, "end": 1, "label": "APP"}]}
    exact = strict_span_metrics(gold_test, {"synthetic": [[0, 1, "APP"]]}, LABELS, ["synthetic"])
    shifted = strict_span_metrics(gold_test, {"synthetic": [[1, 2, "APP"]]}, LABELS, ["synthetic"])
    assert exact["total_true_positives"] == 1 and shifted["total_true_positives"] == 0

    # KD: ignored/padded positions must not influence the objective; teacher is detached.
    torch.manual_seed(7)
    student_logits = torch.randn(2, 4, len(BIO_LABELS), requires_grad=True)
    teacher_logits = torch.randn(2, 4, len(BIO_LABELS), requires_grad=True)
    valid_mask = torch.tensor([[True, False, True, False], [True, True, False, False]])
    kd_1 = temperature_scaled_kd_loss(student_logits, teacher_logits, valid_mask, temperature=2.0)
    altered_teacher = teacher_logits.detach().clone()
    altered_teacher[~valid_mask] = altered_teacher[~valid_mask] * 100.0
    kd_2 = temperature_scaled_kd_loss(student_logits, altered_teacher, valid_mask, temperature=2.0)
    assert torch.allclose(kd_1.detach(), kd_2.detach(), atol=1e-6), (kd_1.item(), kd_2.item())
    kd_1.backward()
    assert torch.isfinite(student_logits.grad).all()
    assert teacher_logits.grad is None, "Teacher should be detached in KD loss."

    print("Synthetic regression tests PASS: exact one-character span, partial-window masking,")
    print("overlap masking, strict off-by-one evaluation, and masked finite KD gradients.")

run_phase2_synthetic_regression_tests()


## 13. Create the train/validation character-boundary datasets and audit coverage

In [ ]:
train_texts = {str(k): TEXTS_BY_ID[str(k)] for k in TRAIN_DOC_IDS}
val_texts = {str(k): TEXTS_BY_ID[str(k)] for k in VAL_DOC_IDS}
train_gold = {str(k): ANNOTATIONS_BY_ID[str(k)] for k in TRAIN_DOC_IDS}
val_gold = {str(k): ANNOTATIONS_BY_ID[str(k)] for k in VAL_DOC_IDS}

train_dataset = CharacterBoundaryWindowDataset(
    TRAIN_DOC_IDS, train_texts, train_gold, tokenizer,
    max_length=MAX_TOKEN_LENGTH, window_overlap=WINDOW_OVERLAP,
)
val_dataset = CharacterBoundaryWindowDataset(
    VAL_DOC_IDS, val_texts, val_gold, tokenizer,
    max_length=MAX_TOKEN_LENGTH, window_overlap=WINDOW_OVERLAP,
)
if not len(train_dataset) or not len(val_dataset):
    raise ValueError("Train/validation window dataset is empty.")
_, val_full_coverage = audit_document_character_coverage(val_texts, val_dataset)
if not val_full_coverage:
    raise RuntimeError("Validation windows do not cover every document character; inspect offset/window logic.")
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    collate_fn=lambda items: collate_character_windows(items, PAD_TOKEN_ID),
)
criterion = WeightedFocalLoss(unique_character_class_weights(train_dataset, DEVICE), gamma=FOCAL_GAMMA).to(DEVICE)
print(f"Train windows: {len(train_dataset):,}; validation windows: {len(val_dataset):,}; max tokens/window: {MAX_TOKEN_LENGTH}")

## 14. Experiment A: train the uploaded hybrid + LoRMA teacher

Only the LoRMA parameters and character head are trainable. The full uploaded hybrid checkpoint remains frozen. The best checkpoint is selected by held-out strict span Macro-F1, not training loss. Gradient accumulation divides by the actual final group size.

In [ ]:
def _amp_context():
    from contextlib import nullcontext
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()


def _save_best(model, path, epoch, metrics, history, extra=None):
    state = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "val_f1": float(metrics["strict_macro_f1"]),
        "val_metrics": {k: metrics[k] for k in ("strict_macro_f1", "macro_precision", "macro_recall", "total_true_positives", "total_false_positives", "total_false_negatives")},
        "history": history,
        "extra": extra or {},
    }
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)


def train_lorma_teacher(model, train_dataset, val_dataset, train_texts, val_texts, train_gold, val_gold):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    best_path = os.path.join(CHECKPOINT_DIR, "best_uploaded_hybrid_lorma_character_boundary.pt")
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                              collate_fn=lambda items: collate_character_windows(items, PAD_TOKEN_ID))
    adapter_parameters = [p for n,p in model.named_parameters() if p.requires_grad and ("lorma_down" in n or "lorma_up" in n)]
    head_parameters = [p for n,p in model.named_parameters() if p.requires_grad and "char_head" in n]
    if not adapter_parameters or not head_parameters:
        raise RuntimeError("LORMA teacher has no trainable adapter or character-head parameters.")
    optimizer = torch.optim.AdamW([
        {"params": adapter_parameters, "lr": LORMA_LR},
        {"params": head_parameters, "lr": CHAR_HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = max(1, math.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
    total_steps = max(1, steps_per_epoch * TEACHER_EPOCHS)
    warmup_steps = max(1, int(total_steps * WARMUP_FRACTION))
    def lr_lambda(step):
        if step < warmup_steps:
            return max(0.05, (step + 1) / warmup_steps)
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return 0.10 + 0.90 * 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    best_f1, best_epoch, stale = -1.0, 0, 0
    history = []
    for epoch in range(1, TEACHER_EPOCHS + 1):
        model.train()
        running_loss, seen = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        epoch_start = time.time()
        n_batches = len(train_loader)
        for step, batch in enumerate(train_loader, start=1):
            inputs = _character_model_inputs(batch, DEVICE)
            targets = batch["char_labels"].to(DEVICE)
            group_start = ((step - 1) // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            actual_group_size = min(GRAD_ACCUM_STEPS, n_batches - group_start)
            with _amp_context():
                logits = model(**inputs)
                raw_loss = criterion(logits, targets)
                loss = raw_loss / actual_group_size
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite teacher loss at epoch {epoch}, step {step}")
            scaler.scale(loss).backward()
            running_loss += float(raw_loss.detach().item())
            seen += 1
            if step % GRAD_ACCUM_STEPS == 0 or step == n_batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
        val = evaluate_character_model(model, val_dataset, val_texts, val_gold)
        row = {"epoch": epoch, "train_loss": running_loss / max(1, seen),
               "val_strict_macro_f1": float(val["strict_macro_f1"]),
               "val_precision": float(val["macro_precision"]), "val_recall": float(val["macro_recall"]),
               "tp": int(val["total_true_positives"]), "fp": int(val["total_false_positives"]),
               "fn": int(val["total_false_negatives"]), "seconds": time.time() - epoch_start}
        history.append(row)
        print(f"[Hybrid-LoRMA] epoch {epoch}/{TEACHER_EPOCHS} loss={row['train_loss']:.4f} strict_macro_F1={row['val_strict_macro_f1']:.4f} P={row['val_precision']:.4f} R={row['val_recall']:.4f} ({row['seconds']:.1f}s)")
        if row["val_strict_macro_f1"] > best_f1 + 1e-7:
            best_f1, best_epoch, stale = row["val_strict_macro_f1"], epoch, 0
            _save_best(model, best_path, epoch, val, history, {"experiment": "UploadedHybrid_LORMA_char_boundary"})
            print("  Saved new best teacher checkpoint:", best_path)
        else:
            stale += 1
            if stale >= TEACHER_PATIENCE:
                print(f"  Early stopping after {stale} non-improving epochs.")
                break
    if not os.path.isfile(best_path):
        raise RuntimeError("Teacher training did not produce a best checkpoint.")
    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model_state_dict"], strict=True)
    model.eval()
    print(f"Restored best teacher epoch={best['epoch']} validation strict macro-F1={best['val_f1']:.4f}")
    return model, best, history

teacher_model, teacher_best_state, teacher_history = train_lorma_teacher(
    teacher_model, train_dataset, val_dataset, train_texts, val_texts, train_gold, val_gold
)
teacher_val_logits = collect_document_character_logits(teacher_model, val_dataset, val_texts, batch_size=BATCH_SIZE)
teacher_raw_val = score_document_logits(teacher_val_logits, val_texts, val_gold, thresholds=None)
print(f"Teacher raw validation strict macro-F1: {teacher_raw_val['strict_macro_f1']:.4f}")

## 15. Experiment B: smaller six-layer student + teacher-student knowledge distillation

The student is initialized from the supplied Phase-1 hybrid weights by selecting six source layers `[0, 2, 4, 6, 8, 11]`. This keeps the global-attention blocks (mapped to student layers `[0, 2, 4, 5]`) and gives the student fewer Transformer blocks. It is trained using both gold character-BIO targets and temperature-softened teacher predictions. The student is full-finetuned, making this materially distinct from teacher-side LoRMA.

In [ ]:
# Distillation student: six hybrid-attention layers initialized from selected pretrained layers.
# Source indices [0,2,4,6,8,11] preserve all four global-attention blocks (mapped to [0,2,4,5]).
STUDENT_SOURCE_LAYER_INDICES = [0, 2, 4, 6, 8, N_LAYER - 1]
if len(set(STUDENT_SOURCE_LAYER_INDICES)) != 6 or max(STUDENT_SOURCE_LAYER_INDICES) >= N_LAYER:
    raise ValueError(f"Cannot construct six-layer student from {N_LAYER}-layer teacher: {STUDENT_SOURCE_LAYER_INDICES}")
STUDENT_GLOBAL_LAYERS = tuple(i for i, src_i in enumerate(STUDENT_SOURCE_LAYER_INDICES) if src_i in GLOBAL_LAYERS)
if not STUDENT_GLOBAL_LAYERS:
    raise RuntimeError("Student has no global-attention layers; invalid distillation mapping.")

# The uploaded Phase-1 class reads these globals while constructing each block.
GLOBAL_ATTENTION_LAYERS = list(STUDENT_GLOBAL_LAYERS)
LOCAL_ATTN_WINDOW = LOCAL_WINDOW_SIZE
student_backbone = HybridLegalLM(
    vocab_size=HYBRID_VOCAB_SIZE, n_embd=N_EMBD, n_layer=len(STUDENT_SOURCE_LAYER_INDICES),
    n_head=N_HEAD, n_kv_head=N_KV_HEAD, max_seq_len=MODEL_CONTEXT_LENGTH,
    ffn_multiplier=FFN_MULTIPLIER, dropout=DROPOUT,
)
# Restore teacher settings for clarity; constructed blocks retain their own attention types.
GLOBAL_ATTENTION_LAYERS = list(GLOBAL_LAYERS)

# Build a complete student state dict, remapping teacher layer weights to student indices.
student_state = student_backbone.state_dict()
for shared_key in ("token_embedding.weight", "lm_head.weight", "final_norm.weight"):
    if shared_key not in phase1_state:
        raise KeyError(f"Hybrid checkpoint is missing required shared tensor {shared_key}")
    if student_state[shared_key].shape != phase1_state[shared_key].shape:
        raise ValueError(f"Student shared tensor shape mismatch for {shared_key}")
    student_state[shared_key] = phase1_state[shared_key].clone()
for new_idx, old_idx in enumerate(STUDENT_SOURCE_LAYER_INDICES):
    old_prefix = f"blocks.{old_idx}."
    new_prefix = f"blocks.{new_idx}."
    mapped = 0
    for key, value in phase1_state.items():
        if key.startswith(old_prefix):
            target_key = new_prefix + key[len(old_prefix):]
            if target_key not in student_state or student_state[target_key].shape != value.shape:
                raise ValueError(f"Cannot map hybrid tensor {key} -> {target_key}")
            student_state[target_key] = value.clone()
            mapped += 1
    if mapped == 0:
        raise KeyError(f"No pretrained hybrid tensors found for source layer {old_idx}")
student_backbone.load_state_dict(student_state, strict=True)
student_model = HybridCharacterNERModel(
    backbone=student_backbone, char_vocab_size=CHAR_VOCAB_SIZE, num_labels=len(BIO_LABELS)
).to(DEVICE)

student_total = sum(p.numel() for p in student_model.parameters())
teacher_total = sum(p.numel() for p in teacher_model.parameters())
print("Student source layers:", STUDENT_SOURCE_LAYER_INDICES)
print("Student global layers:", STUDENT_GLOBAL_LAYERS)
print(f"Teacher total parameters incl. head: {teacher_total:,} ({teacher_total/1e6:.2f}M)")
print(f"Student total parameters incl. head: {student_total:,} ({student_total/1e6:.2f}M)")
if student_total >= teacher_total:
    raise RuntimeError("Distillation student is not smaller than the teacher.")


def train_distilled_student(student, teacher, train_dataset, val_dataset, train_texts, val_texts, train_gold, val_gold):
    best_path = os.path.join(CHECKPOINT_DIR, "best_uploaded_hybrid_distilled_student_character_boundary.pt")
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                              collate_fn=lambda items: collate_character_windows(items, PAD_TOKEN_ID))
    head_params = list(student.char_head.parameters())
    backbone_params = [p for p in student.backbone.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": STUDENT_BACKBONE_LR},
        {"params": head_params, "lr": CHAR_HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = max(1, math.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
    total_steps = max(1, steps_per_epoch * STUDENT_EPOCHS)
    warmup_steps = max(1, int(total_steps * WARMUP_FRACTION))
    def lr_lambda(step):
        if step < warmup_steps:
            return max(0.05, (step + 1) / warmup_steps)
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return 0.10 + 0.90 * 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    teacher.eval()
    best_f1, best_epoch, stale = -1.0, 0, 0
    history = []
    for epoch in range(1, STUDENT_EPOCHS + 1):
        student.train()
        running_total, running_hard, running_kd, seen = 0.0, 0.0, 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        epoch_start = time.time()
        n_batches = len(train_loader)
        for step, batch in enumerate(train_loader, start=1):
            inputs = _character_model_inputs(batch, DEVICE)
            targets = batch["char_labels"].to(DEVICE)
            # Distill only valid characters with supervised labels. Padding and
            # ignored partial/overlapping spans must not contribute to KL.
            char_mask = inputs["char_mask"].bool() & targets.ne(IGNORE_INDEX)
            group_start = ((step - 1) // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            actual_group_size = min(GRAD_ACCUM_STEPS, n_batches - group_start)
            with torch.no_grad():
                with _amp_context():
                    teacher_logits = teacher(**inputs).float()
            with _amp_context():
                student_logits = student(**inputs)
                hard_loss = criterion(student_logits, targets)
                temperature = KD_TEMPERATURE
                kd_loss = temperature_scaled_kd_loss(
                    student_logits, teacher_logits, char_mask, temperature=temperature
                )
                raw_loss = KD_HARD_WEIGHT * hard_loss + KD_SOFT_WEIGHT * kd_loss
                loss = raw_loss / actual_group_size
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite student KD loss at epoch {epoch}, step {step}")
            scaler.scale(loss).backward()
            running_total += float(raw_loss.detach().item())
            running_hard += float(hard_loss.detach().item())
            running_kd += float(kd_loss.detach().item())
            seen += 1
            if step % GRAD_ACCUM_STEPS == 0 or step == n_batches:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(student.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
        val = evaluate_character_model(student, val_dataset, val_texts, val_gold)
        row = {"epoch": epoch, "train_loss": running_total/max(1,seen),
               "hard_loss": running_hard/max(1,seen), "kd_loss": running_kd/max(1,seen),
               "val_strict_macro_f1": float(val["strict_macro_f1"]),
               "val_precision": float(val["macro_precision"]), "val_recall": float(val["macro_recall"]),
               "tp": int(val["total_true_positives"]), "fp": int(val["total_false_positives"]),
               "fn": int(val["total_false_negatives"]), "seconds": time.time()-epoch_start}
        history.append(row)
        print(f"[Hybrid-KD student] epoch {epoch}/{STUDENT_EPOCHS} total={row['train_loss']:.4f} hard={row['hard_loss']:.4f} kd={row['kd_loss']:.4f} strict_macro_F1={row['val_strict_macro_f1']:.4f}")
        if row["val_strict_macro_f1"] > best_f1 + 1e-7:
            best_f1, best_epoch, stale = row["val_strict_macro_f1"], epoch, 0
            _save_best(student, best_path, epoch, val, history, {
                "experiment": "UploadedHybrid_KD_6layer_student", "source_layer_indices": STUDENT_SOURCE_LAYER_INDICES,
                "kd_temperature": KD_TEMPERATURE, "hard_weight": KD_HARD_WEIGHT, "soft_weight": KD_SOFT_WEIGHT,
            })
            print("  Saved new best distilled student checkpoint:", best_path)
        else:
            stale += 1
            if stale >= STUDENT_PATIENCE:
                print("  Early stopping student after non-improving epochs.")
                break
    if not os.path.isfile(best_path):
        raise RuntimeError("Student distillation did not produce a best checkpoint.")
    best = torch.load(best_path, map_location=DEVICE, weights_only=False)
    student.load_state_dict(best["model_state_dict"], strict=True)
    student.eval()
    print(f"Restored best student epoch={best['epoch']} validation strict macro-F1={best['val_f1']:.4f}")
    return student, best, history

student_model, student_best_state, student_history = train_distilled_student(
    student_model, teacher_model, train_dataset, val_dataset, train_texts, val_texts, train_gold, val_gold
)
student_val_logits = collect_document_character_logits(student_model, val_dataset, val_texts, batch_size=BATCH_SIZE)
student_raw_val = score_document_logits(student_val_logits, val_texts, val_gold, thresholds=None)
print(f"Distilled student raw validation strict macro-F1: {student_raw_val['strict_macro_f1']:.4f}")

## 16. Validation-only confidence calibration

The coordinate search only uses held-out validation logits and gold spans. We preserve both raw and calibrated results and keep raw thresholds as a fallback when calibration does not help on validation. The validation score after calibration is a model-selection score, not an unbiased generalization estimate.

In [ ]:
def calibrate_label_biases(logits_by_doc, texts_by_id, gold_by_id, passes=2):
    raw_metrics = score_document_logits(logits_by_doc, texts_by_id, gold_by_id, thresholds=None)
    thresholds = {label: 0.0 for label in LABELS}
    best_metrics = raw_metrics
    # Signed per-label logit biases can either promote rare/under-recalled labels
    # (negative values) or suppress false positives (positive values).
    candidates = [-1.0, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0, 1.5]
    for pass_idx in range(passes):
        changed = False
        for label in LABELS:
            current_bias = thresholds[label]
            best_candidate = current_bias
            current_score = best_metrics["strict_macro_f1"]
            for candidate in candidates:
                trial = dict(thresholds)
                trial[label] = candidate
                trial_metrics = score_document_logits(logits_by_doc, texts_by_id, gold_by_id, thresholds=trial)
                if trial_metrics["strict_macro_f1"] > current_score + 1e-7:
                    current_score, best_candidate, best_metrics = trial_metrics["strict_macro_f1"], candidate, trial_metrics
            if best_candidate != current_bias:
                changed = True
            thresholds[label] = best_candidate
        if not changed:
            break
    if best_metrics["strict_macro_f1"] + 1e-8 < raw_metrics["strict_macro_f1"]:
        thresholds = {label: 0.0 for label in LABELS}
        best_metrics = raw_metrics
    return thresholds, raw_metrics, best_metrics

teacher_thresholds, teacher_raw_val, teacher_calibrated_val = calibrate_label_biases(
    teacher_val_logits, val_texts, val_gold
)
student_thresholds, student_raw_val, student_calibrated_val = calibrate_label_biases(
    student_val_logits, val_texts, val_gold
)

def _write_json(path, payload):
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)

validation_report = {
    "split": {"train_documents": len(TRAIN_DOC_IDS), "validation_documents": len(VAL_DOC_IDS)},
    "teacher_hybrid_lorma": {
        "best_epoch": int(teacher_best_state["epoch"]),
        "raw_validation": {k: teacher_raw_val[k] for k in ("strict_macro_f1", "macro_precision", "macro_recall", "total_true_positives", "total_false_positives", "total_false_negatives")},
        "calibrated_validation": {k: teacher_calibrated_val[k] for k in ("strict_macro_f1", "macro_precision", "macro_recall", "total_true_positives", "total_false_positives", "total_false_negatives")},
        "thresholds": teacher_thresholds,
        "per_label_calibrated": teacher_calibrated_val["per_label"],
    },
    "student_hybrid_kd": {
        "best_epoch": int(student_best_state["epoch"]),
        "raw_validation": {k: student_raw_val[k] for k in ("strict_macro_f1", "macro_precision", "macro_recall", "total_true_positives", "total_false_positives", "total_false_negatives")},
        "calibrated_validation": {k: student_calibrated_val[k] for k in ("strict_macro_f1", "macro_precision", "macro_recall", "total_true_positives", "total_false_positives", "total_false_negatives")},
        "thresholds": student_thresholds,
        "per_label_calibrated": student_calibrated_val["per_label"],
    },
}
_write_json(os.path.join(OUTPUT_DIR, "validation_report.json"), validation_report)
print(json.dumps(validation_report, indent=2))

## 17. Select one model on validation and create exactly one submission ZIP

Both the LoRMA teacher and distilled student are evaluated on the same held-out validation split. This cell chooses the higher validation Strict Macro-F1 after validation-only calibration (teacher wins ties), uses the matching model and matching thresholds, and packages exactly one ZIP. It deletes only stale `submission_*.zip` files inside this experiment's dedicated output directory. Nothing is submitted to Codabench automatically.

In [ ]:
# Choose one candidate using the held-out validation result, then package a single ZIP.
# The chosen model and its calibration thresholds remain paired; no cross-model threshold reuse.

test_texts = {str(k): v for k, v in TEST_TEXTS.items()}
test_dataset = CharacterBoundaryWindowDataset(
    document_ids=list(test_texts.keys()), text_by_id=test_texts, entities_by_id={}, tokenizer_obj=tokenizer,
    max_length=MAX_TOKEN_LENGTH, window_overlap=WINDOW_OVERLAP,
)
_, test_full_coverage = audit_document_character_coverage(test_texts, test_dataset)
if not test_full_coverage:
    raise RuntimeError("Test character coverage is incomplete; refusing to create a submission.")

model_candidates = [
    {
        "name": "lorma_teacher", "model": teacher_model, "thresholds": teacher_thresholds,
        "calibrated_f1": float(teacher_calibrated_val["strict_macro_f1"]),
        "raw_f1": float(teacher_raw_val["strict_macro_f1"]),
        "best_epoch": int(teacher_best_state["epoch"]),
    },
    {
        "name": "distilled_student", "model": student_model, "thresholds": student_thresholds,
        "calibrated_f1": float(student_calibrated_val["strict_macro_f1"]),
        "raw_f1": float(student_raw_val["strict_macro_f1"]),
        "best_epoch": int(student_best_state["epoch"]),
    },
]
# Stable tie-break: prefer the teacher when validation scores are equal.
selected_candidate = max(
    model_candidates, key=lambda item: (item["calibrated_f1"], item["raw_f1"], item["name"] == "lorma_teacher")
)
selected_model = selected_candidate["model"]
selected_thresholds = selected_candidate["thresholds"]
selected_model_name = selected_candidate["name"]
print("Selected candidate:", selected_model_name)
print("Selected best epoch:", selected_candidate["best_epoch"])
print("Selected raw validation strict Macro-F1:", f"{selected_candidate['raw_f1']:.5f}")
print("Selected calibrated validation strict Macro-F1:", f"{selected_candidate['calibrated_f1']:.5f}")

# Generate predictions from the same model whose validation outputs produced these thresholds.
test_logits = collect_document_character_logits(
    selected_model, test_dataset, test_texts, batch_size=BATCH_SIZE
)
prediction = {
    doc_id: decode_character_bio_logits(logits, thresholds=selected_thresholds)
    for doc_id, logits in test_logits.items()
}
for doc_id in test_texts:
    prediction.setdefault(doc_id, [])
if set(prediction) != set(test_texts):
    raise ValueError("Test document IDs mismatch; refusing to package submission.")

for doc_id, entities in prediction.items():
    seen_spans = set()
    for start, end, label in entities:
        if label not in LABELS or not (0 <= int(start) < int(end) <= len(test_texts[doc_id])):
            raise ValueError(f"Invalid predicted span in {doc_id}: {(start, end, label)}")
        triple = (int(start), int(end), str(label))
        if triple in seen_spans:
            raise ValueError(f"Duplicate predicted span in {doc_id}: {triple}")
        seen_spans.add(triple)

os.makedirs(OUTPUT_DIR, exist_ok=True)
# Remove stale archives created by earlier versions in this same dedicated experiment folder.
for stale_name in os.listdir(OUTPUT_DIR):
    if stale_name.startswith("submission_") and stale_name.endswith(".zip"):
        os.remove(os.path.join(OUTPUT_DIR, stale_name))

prediction_path = os.path.join(OUTPUT_DIR, "prediction.json")
submission_zip_path = os.path.join(OUTPUT_DIR, "submission_uploaded_hybrid_best_validation.zip")
_write_json(prediction_path, prediction)
with zipfile.ZipFile(submission_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(prediction_path, arcname="prediction.json")
with zipfile.ZipFile(submission_zip_path, "r") as archive:
    if archive.namelist() != ["prediction.json"]:
        raise RuntimeError("Submission ZIP must contain prediction.json at archive root only.")
    packaged_prediction = json.loads(archive.read("prediction.json").decode("utf-8"))
if set(packaged_prediction) != set(test_texts):
    raise RuntimeError("Packaged document IDs do not match fold-3 test IDs.")
zip_files = [name for name in os.listdir(OUTPUT_DIR) if name.startswith("submission_") and name.endswith(".zip")]
if zip_files != [os.path.basename(submission_zip_path)]:
    raise RuntimeError(f"Expected exactly one submission ZIP in {OUTPUT_DIR}; found {zip_files}")

selection_report = {
    "selected_model": selected_model_name,
    "selected_best_epoch": selected_candidate["best_epoch"],
    "selected_raw_validation_strict_macro_f1": selected_candidate["raw_f1"],
    "selected_calibrated_validation_strict_macro_f1": selected_candidate["calibrated_f1"],
    "all_candidates": [
        {"name": c["name"], "best_epoch": c["best_epoch"], "raw_validation_strict_macro_f1": c["raw_f1"],
         "calibrated_validation_strict_macro_f1": c["calibrated_f1"]} for c in model_candidates
    ],
    "prediction_json": prediction_path,
    "submission_zip": submission_zip_path,
    "test_documents": len(test_texts),
    "predicted_spans": sum(len(v) for v in prediction.values()),
    "note": "Candidate selection and threshold calibration use the held-out validation split; the displayed validation score is for model selection and is not an unbiased test estimate."
}
_write_json(os.path.join(OUTPUT_DIR, "selected_submission_report.json"), selection_report)
print("Created exactly one final submission ZIP:", submission_zip_path)
print("Prediction JSON:", prediction_path)
print("Test documents:", len(prediction), "| entities:", sum(len(v) for v in prediction.values()))
print(json.dumps(selection_report, indent=2))
print("No automatic Codabench submission was performed.")


## 18. Experiment manifest and status

This notebook runs LoRMA and teacher-student distillation using the exact hybrid architecture in your uploaded file. It does not, by itself, satisfy the course-wide requirement to experiment with all four Phase-1 architectures.

In [ ]:
manifest = {
    "experiment_family": "CS781_Phase2_LNER_Uploaded_Hybrid",
    "phase1_architecture": "HybridLegalLM copied from the uploaded v4(1).ipynb; decoder-only global/local attention, RMSNorm + RoPE + GQA; embedded checkpoint metadata preserved",
    "source_v4_checkpoint": HYBRID_CHECKPOINT_PATH,
    "tokenizer_path": BPE_PATH,
    "data_sources": PHASE2_FILES,
    "split_seed": SEED,
    "train_documents": len(TRAIN_DOC_IDS),
    "validation_documents": len(VAL_DOC_IDS),
    "test_documents": len(TEST_TEXTS),
    "adaptation_1": {
        "name": "LoRMA-style multiplicative low-rank adaptation on q_proj/v_proj plus character boundary head",
        "rank": LORMA_RANK, "alpha": LORMA_ALPHA, "dropout": LORMA_DROPOUT,
        "best_epoch": int(teacher_best_state["epoch"]),
        "raw_validation_strict_macro_f1": float(teacher_raw_val["strict_macro_f1"]),
        "calibrated_validation_strict_macro_f1": float(teacher_calibrated_val["strict_macro_f1"]),
        "trainable_parameter_count": int(teacher_trainable_count),
    },
    "adaptation_2": {
        "name": "6-layer hybrid student initialized from selected Phase-1 hybrid layers and trained with hard labels + teacher-logit distillation",
        "source_layer_indices": STUDENT_SOURCE_LAYER_INDICES,
        "global_attention_layers": STUDENT_GLOBAL_LAYERS,
        "temperature": KD_TEMPERATURE, "hard_weight": KD_HARD_WEIGHT, "soft_weight": KD_SOFT_WEIGHT,
        "best_epoch": int(student_best_state["epoch"]),
        "raw_validation_strict_macro_f1": float(student_raw_val["strict_macro_f1"]),
        "calibrated_validation_strict_macro_f1": float(student_calibrated_val["strict_macro_f1"]),
        "total_parameters": int(student_total),
        "teacher_total_parameters": int(teacher_total),
    },
    "validation_report_path": os.path.join(OUTPUT_DIR, "validation_report.json"),
    "submission_archive": os.path.join(OUTPUT_DIR, "submission_uploaded_hybrid_best_validation.zip"),
    "submission_policy": "Exactly one ZIP is produced; selected candidate is the higher validation Strict Macro-F1 after non-degrading validation calibration.",
    "disclaimer": "No leaderboard score is implied by local validation. Submit one archive at a time and compare returned Codabench scores.",
}
manifest_path = os.path.join(OUTPUT_DIR, "experiment_manifest_uploaded_hybrid_lorma_kd.json")
_write_json(manifest_path, manifest)
print(json.dumps(manifest, indent=2))
print("Manifest:", manifest_path)

## Before using the single submission

1. **Device fix:** LoRMA down/up projections inherit the wrapped pretrained Q/V projection device and dtype. The injection cell asserts adapter/base device equality and checks function-preserving identity initialization. Confirm the strict checkpoint load passed and inspect `CHECKPOINT_ARCHITECTURE_LABEL`. The uploaded `v4(1).ipynb` itself saves `hybrid_model.pt` and labels its architecture V2 / `lightweight_hybrid_decoder`; this experiment intentionally preserves that exact model code and metadata.
2. Review the per-label validation report and `selected_submission_report.json`. The validation score is for model selection, not an unbiased test estimate.
3. The finalization cell selects either the LoRMA teacher or the distilled student using validation Strict Macro-F1 after calibration, then creates **exactly one** `submission_uploaded_hybrid_best_validation.zip`. Submit only that ZIP, after you have personally checked the output.
4. No full-data refit is performed after calibration. The exact model used for test inference is the same model whose held-out validation logits generated the chosen thresholds.
5. This notebook has not been run against your actual Kaggle checkpoint here, and no leaderboard score is claimed.

### Method references

- Bihany, Patel & Modi (2025), *LoRMA: Low-Rank Multiplicative Adaptation for LLMs*, Findings of ACL 2025: https://aclanthology.org/2025.findings-acl.527/ . LoRMA uses low-rank multiplicative updates, distinct from standard additive LoRA.
- Hinton, Vinyals & Dean (2015), *Distilling the Knowledge in a Neural Network*: https://arxiv.org/abs/1503.02531 . The student uses teacher soft predictions alongside gold labels.
- Su et al. (2022), *Global Pointer: Novel Efficient Span-based Approach for Named Entity Recognition*: https://arxiv.org/abs/2208.03054 . This motivates considering exact spans; the present notebook retains character BIO modelling rather than replacing the head with GlobalPointer.
